<!-- derived-from: ../../02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98/02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98-LangChain-%E8%BF%9B%E9%98%B6%E7%89%88.ipynb -->
> English mirror, derived from the Chinese master: [02_langchain-advanced.ipynb](../../02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98/02_%E7%BB%BC%E5%90%88%E5%AE%9E%E6%88%98-LangChain-%E8%BF%9B%E9%98%B6%E7%89%88.ipynb). The Chinese notebook is the source of truth.
>
> Cell outputs are intentionally empty in this mirror: they hold Chinese corpus text, and this course never hand-writes model output. Run the notebook on the vLLM + Xinference + Milvus stack to populate them.
>
> Chinese literals that the code depends on (corpus file paths, regexes over the Chinese documents, demo retrieval queries, section labels) are kept verbatim: the knowledge base itself is Chinese, so translating them would change the behaviour being taught.
> **Runtime preparation**: before running anything else, do two basic setup steps.
> 1. `warnings.filterwarnings("ignore")`: silence runtime warnings so deprecation warnings from legacy LangChain APIs (such as `langchain_classic`) do not distract from the lesson.
> 2. Clear the `HTTP_PROXY` / `HTTPS_PROXY` / `ALL_PROXY` environment variables: prevent proxy settings on the host or inside the container from breaking access to vLLM, Xinference, Milvus and the other services.


In [ ]:
import warnings
warnings.filterwarnings("ignore")


import os
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy"
]:
    os.environ.pop(k, None)

# About this project

This project is a **hands-on introduction to Retrieval-Augmented Generation (RAG)** for beginners. LangChain is the core framework, and the notebook walks you through building a local knowledge-base Q&A system from scratch: RAG fundamentals, environment setup, document loading and splitting, vectorisation and vector storage, retrieval strategies, LLM generation, and a simple LLM-as-Judge evaluation.

## 1. Project overview

The notebook follows the pipeline *data -> index -> retrieval -> generation -> evaluation*, so learners build an end-to-end view of a RAG system. It covers:

1. **Environment setup**: creating the uv virtual environment, installing dependencies, configuring the vLLM / Xinference model services;
2. **Knowledge-base construction**: load local PDF and Markdown documents with LangChain, split the text, embed the chunks, store them in a vector database;
3. **Retrieval and generation**: recall relevant chunks by vector search, then have the LLM generate a grounded answer;
4. **Evaluation loop**: use LLM-as-Judge to compare the RAG answer and the baseline answer on faithfulness, completeness and relevance;
5. **Where to go next**: a short intro to systematic evaluation frameworks such as Ragas as a reference for future optimisation.

After finishing this project you will be able to build a runnable, evaluable local knowledge-base Q&A system on your own, and you will understand what each step of the RAG pipeline does and where to tune it.


## 2. Key libraries and environment notes


> **Environment note**: this notebook runs in the `env_rag_uv` virtual environment created with `uv`, on Python 3.12, and the Jupyter kernel is also named `env_rag_uv`. The table below lists the core dependencies, what they are for, and the version actually installed. When running, the code cell below prints the real installed version of every library in your environment.


| Library | Version | Purpose |
|---|---|---|
| `langchain` | 1.4.7 | Main LangChain framework: chains, prompts, model interfaces and other core capabilities |
| `langchain-core` | 1.7.0 | LangChain core abstractions and runtime (Messages, Runnables, output parsers, ...) |
| `langchain-community` | 0.4.2 | Community integrations (document loaders such as `PyMuPDFLoader`) |
| `langchain-openai` | 1.7.2 | OpenAI-compatible interface; used here for the vLLM chat model and the Xinference embedding model |
| `langchain-milvus` | 0.4.0 | LangChain wrapper for the Milvus vector database (requires pymilvus 3.x) |
| `langchain-classic` | 1.0.8 | Legacy API compatibility (`RetrievalQA`, `ConversationalRetrievalChain`) |
| `langchain-text-splitters` | 1.1.3 | Text splitters (`RecursiveCharacterTextSplitter`, ...) |
| `langchain-cohere` | not installed | Cohere Rerank cloud reranking (optional, auto-skipped without a key) |
| `pymilvus` | 3.0.2 | Milvus Python SDK, used to connect to and operate the vector database |
| `PyMuPDF` | 2.0 | PDF parser, the underlying dependency of `PyMuPDFLoader` |
| `rank-bm25` | 0.2.2 | Hand-written BM25 keyword retrieval for teaching |
| `xinference-client` | 4.0.0 | Native Xinference REST client (optional; embedding and rerank go through the OpenAI-compatible interface) |
| `python-dotenv` | 1.2.4 | Read service URLs and API keys from the `.env` file |
| `httpx` | 0.28.1 | HTTP client, used for vLLM / Xinference rerank / Jina rerank calls |
| `openai` | 3.37.0 | OpenAI SDK, required for OpenAI-compatible endpoints |
| `pandas` | 2.3.3 | Data handling and display |
| `numpy` | 2.5.4 | Numerical computing |
| `ipykernel` | 7.4.0 | Jupyter kernel needed to run the notebook |


In [ ]:
# Print the actually installed version of the key libraries in the current env_rag_uv environment
from importlib.metadata import version, PackageNotFoundError

packages = [
    ("langchain", "LangChain core framework"),
    ("langchain-core", "LangChain core abstractions"),
    ("langchain-community", "Community components and loaders"),
    ("langchain-openai", "OpenAI-compatible interface (vLLM / Xinference)"),
    ("langchain-milvus", "Milvus vector store wrapper"),
    ("langchain-classic", "Legacy Chain API compatibility"),
    ("langchain-text-splitters", "Text splitters"),
    ("langchain-cohere", "Cohere Rerank (optional)"),
    ("pymilvus", "Milvus Python SDK"),
    ("PyMuPDF", "PDF parsing"),
    ("rank-bm25", "BM25 keyword retrieval (teaching build)"),
    ("xinference-client", "Native Xinference client (optional)"),
    ("python-dotenv", ".env configuration reader"),
    ("httpx", "HTTP client"),
    ("openai", "OpenAI SDK"),
    ("pandas", "Data handling"),
    ("numpy", "Numerical computing"),
    ("ipykernel", "Jupyter Kernel"),
]

print(f"{'name':<35} {'version':<15} {'purpose'}")
print("-" * 80)
for name, desc in packages:
    try:
        ver = version(name)
    except PackageNotFoundError:
        ver = "not installed"
    print(f"{name:<35} {ver:<15} {desc}")

## 3. Unified model configuration

This notebook uses one shared set of three model configurations:

- **Chat model**: Qwen3 served by vLLM, called over the OpenAI-compatible protocol with `langchain.chat_models.init_chat_model`;
- **Embedding model**: Qwen3 Embedding served by Xinference, called through its OpenAI-compatible interface with `langchain_openai.OpenAIEmbeddings`;
- **Reranker model**: Qwen3 Reranker served by Xinference, called through the OpenAI-compatible `{XINFERENCE_OPENAI_BASE_URL}/rerank` endpoint.

The addresses, model names and keys of all three services live in the repository root `.env`; the code only reads environment variables and never hard-codes an IP or a key.
The cells below define `my_llm`, `my_emb` and the `RERANKER_BASE_URL` / `RERANKER_MODEL` constants once.
Later chapters sometimes redefine the same configuration inside their own cells so each section can run standalone.


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

# Read the repository root .env (searching upwards automatically when the notebook lives in a subdirectory)
_ = load_dotenv(find_dotenv())

# -- 1. Chat model configuration (vLLM, OpenAI-compatible interface) ----------------
my_llm = init_chat_model(
    model=os.environ["vLLM_QWEN_MODEL_ID"],
    model_provider="openai",
    base_url=os.environ["vLLM_QWEN_BASE_URL"],
    api_key=os.environ["vLLM_QWEN_API_KEY"],
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=3000,
)

# -- 2. Embedding model configuration (Xinference, OpenAI-compatible interface) ------
my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

# -- 3. Reranker model configuration (Xinference, /v1/rerank) -----------------------
RERANKER_BASE_URL = os.environ["XINFERENCE_OPENAI_BASE_URL"]
RERANKER_MODEL = os.environ["XINFERENCE_RERANKER_MODEL_ID"]

# -- 4. Vector database address ------------------------------------------------------
MILVUS_URI = os.environ["MILVUS_URI"]

print("LLM       :", os.environ["vLLM_QWEN_MODEL_ID"], "@", os.environ["vLLM_QWEN_BASE_URL"])
print("Embedding :", os.environ["XINFERENCE_EMBEDDING_MODEL_ID"], "@", os.environ["XINFERENCE_OPENAI_BASE_URL"])
print("Reranker  :", RERANKER_MODEL)
print("Milvus    :", MILVUS_URI)

In [ ]:
import os
import httpx

# Embedding self-check: confirm the vector dimension matches what Xinference returns
vector = my_emb.embed_query("What is artificial intelligence")
print("vector dimension:", len(vector))
print("first 5 dimensions:", [round(v, 4) for v in vector[:5]])

# Rerank self-check: call Xinference's OpenAI-compatible /rerank endpoint
response = httpx.post(
    RERANKER_BASE_URL.rstrip("/") + "/rerank",
    headers={"Authorization": "Bearer " + os.environ["XINFERENCE_API_KEY"]},
    json={
        "model": RERANKER_MODEL,
        "query": "What is artificial intelligence",
        "documents": [
            "Artificial intelligence is the field that studies how to make machines simulate human intelligence",
            "The weather is fine today and it is a good day for a walk",
        ],
        "top_n": 2,
    },
    timeout=60,
    trust_env=False,
)
response.raise_for_status()
for item in response.json()["results"]:
    print(f"index={item['index']}  score={item['relevance_score']:.4f}")

# Data preprocessing


## 1. Simple PDF preprocessing

To build our local knowledge base we must process local documents stored in various formats: read the documents, turn their content into embeddings with the embedding method described earlier, and build a vector database from them. The following sections start from concrete examples and show how to process local documents.


### 1.1 Reading the data
If you have no knowledge-base files yet, put the corpus sources into the knowledge_db directory.


In [ ]:
# !cp -r /workspace/DataSet/rag2026/VMAX-S/   /workspace/llm-rag/knowledge_path/

PDF documents as an example:   
Use LangChain's PyMuPDFLoader to read the PDF files in the knowledge base. PyMuPDFLoader is the fastest PDF parser available: its result carries detailed metadata about the PDF and each page, and it returns one document per page.


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

# Create a PyMuPDFLoader instance, passing the path of the PDF document to load
loader = PyMuPDFLoader("../knowledge_path/VMAX-S/ZXVMAX-S\uff08V6.23\uff09\u4ea7\u54c1\u63cf\u8ff0\uff08\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1\uff09.pdf")

# Call the load method of the PyMuPDFLoader class to load the PDF file
pdf_pages = loader.load()

The loaded document is stored in the `pdf_pages` variable:
- `pdf_pages` has type `List`
- printing `len(pdf_pages)` tells you how many pages the PDF contains


In [ ]:
print(f"loaded variable type: {type(pdf_pages)}",  f"this PDF contains {len(pdf_pages)} pages")

Each element of `pdf_pages` is one document of type `langchain_core.documents.base.Document`, which has two attributes:
- `page_content` holds the content of the document.
- `meta_data` holds descriptive metadata about the document.


In [ ]:
pdf_page = pdf_pages[20]
print(f"type of each element: {type(pdf_page)}.",
    f"descriptive metadata of this document: {pdf_page.metadata}",
    f"content of this document:\n{pdf_page.page_content}",
    sep="\n------\n")

In [ ]:
# suppose the NLTK punkt resource is missing - punkt is the tokenizer NLTK needs. Fix it by downloading and installing punkt exactly as the error message suggests.
# import nltk
# nltk.download('punkt', quiet=True)

### 1.2 Cleaning

The simple version only does generic whitespace normalisation: collapse runs of newlines and spaces into a single space. It is easy to understand, but it does not recognise headers and footers, headings, lists or the hard line breaks introduced by PDF layout. Treat it as the baseline for the optimised version.


In [ ]:
import re
from langchain_core.documents import Document

def simple_clean_document(doc: Document) -> Document:
    text = re.sub(r"\s+", " ", doc.page_content).strip()
    metadata = dict(doc.metadata)
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["preprocess_version"] = "simple"
    return Document(page_content=text, metadata=metadata)

simple_cleaned_pages = [
    simple_clean_document(page) for page in pdf_pages if page.page_content.strip()
]
print(simple_cleaned_pages[20].page_content[:500])

### 1.3 Splitting

The simple version splits purely by fixed character length, with no awareness of chapter boundaries.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

simple_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
)
simple_split_docs = simple_text_splitter.split_documents(simple_cleaned_pages)
for index, chunk in enumerate(simple_split_docs):
    chunk.metadata["chunk_index"] = index

print(f"number of chunks after simple preprocessing: {len(simple_split_docs)}")

## 2. Optimised PDF preprocessing

The previous section showed basic PDF loading and page structure. This section presents an optimised pipeline aimed at the ZXVMAX-S product manual: first restore the document structure and strip noise such as headers and footers, then build semantic boundaries from chapter titles, then split only the chapters that are too long, while keeping source file, page number, business type, chapter title and chunk index as metadata.


### 2.1 Reading the data

The optimised version reads the same PDF, so the comparison below reflects only the difference in preprocessing strategy.


In [ ]:
optimized_loader = PyMuPDFLoader(
    "../knowledge_path/VMAX-S/ZXVMAX-S\uff08V6.23\uff09\u4ea7\u54c1\u63cf\u8ff0\uff08\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1\uff09.pdf"
)
optimized_pdf_pages = optimized_loader.load()
print(f"pages read by the optimised version: {len(optimized_pdf_pages)}")

### 2.2 Cleaning


Cleaning a PDF is not just deleting newlines and spaces; the goal is to recover the document's original semantic structure. For the ZXVMAX-S product manual used here, the things that matter are headers and footers, standalone page numbers, the hard line breaks produced by PDF layout, and private-use-area bullet characters such as `.`.  

This section works in three steps: normalise characters first, then filter noise by rule, then restore paragraph structure. Headings, lists, English terms and necessary spaces are preserved.


Define reusable cleaning rules


In [ ]:
import re
import unicodedata
from pathlib import Path
from langchain_core.documents import Document

# ZXVMAX-S fixed page footer, for example: SJ-... | 2023-03-30 (R1.0)
FOOTER_PATTERN = re.compile(
    "^SJ-\\d{14,}-\\d+\\s*\\|\\s*\\d{4}-\\d{2}-\\d{2}\uff08R\\d+\\.\\d+\uff09$"
)
PAGE_NUMBER_PATTERN = re.compile(r"^\d{1,4}$")
HEADING_PATTERN = re.compile(
    "^(?:\u7b2c[\u4e00\u4e8c\u4e09\u56db\u4e94\u516d\u4e03\u516b\u4e5d\u5341\u767e]+\u7ae0|\\d+(?:\\.\\d+)*)(?:\\s+|\\u00a0).+"
)
BULLET_PATTERN = re.compile(r"^[\uf06c•▪▫▶●○]\s*")
REPEATED_HEADER_PATTERN = re.compile(
    "^ZXVMAX-S(?: \u591a\u7ef4\u4ef7\u503c\u5206\u6790\u7cfb\u7edf)? \u4ea7\u54c1\u63cf\u8ff0\uff08.+\u4e1a\u52a1\uff09$"
)

In [ ]:
def normalize_line(line: str) -> str:
    """Normalise Unicode, whitespace and bullet characters without deleting meaningful spaces."""
    # NFC only normalises Unicode composition forms; it never turns Chinese punctuation into ASCII punctuation
    line = unicodedata.normalize("NFC", line)
    line = line.replace("\u00a0", " ").replace("\u3000", " ")
    line = re.sub(r"[ \t]+", " ", line).strip()
    if BULLET_PATTERN.match(line):
        line = BULLET_PATTERN.sub("- " , line)
    return line


def should_join(previous: str, current: str) -> bool:
    """Decide whether a line break comes from PDF layout; headings, list items and complete sentences are never merged."""
    if not previous or not current:
        return False
    if current.startswith("- ") or previous.startswith("- "):
        return False
    if HEADING_PATTERN.match(previous) or HEADING_PATTERN.match(current):
        return False
    # unnumbered sub-headings in the product manual are usually short, such as "fault tolerance" or "collection layer"
    if len(previous) <= 20 and not previous.endswith(("\uff0c", "\u3001", "\uff1a", "\uff1b")):
        return False
    if previous.endswith(("\u3002", "\uff01", "\uff1f", "\uff1b", "\uff1a", "\uff09", "\u3011")):
        return False
    return True


def clean_pdf_document(doc: Document) -> Document:
    """Clean one PDF page while fully preserving the source and page metadata."""
    raw_text = doc.page_content.replace("\r\n", "\n").replace("\r", "\n")
    lines = []
    pending_bullet = False

    for raw_line in raw_text.split("\n"):
        line = normalize_line(raw_line)
        if not line:
            continue
        # bullet characters often occupy a line of their own in a PDF; merge them with the list title on the next line
        if line == "- ":
            pending_bullet = True
            continue
        if pending_bullet:
            line = "- " + line
            pending_bullet = False
        if FOOTER_PATTERN.match(line) or PAGE_NUMBER_PATTERN.match(line):
            continue
        if REPEATED_HEADER_PATTERN.match(line):
            continue
        lines.append(line)

    paragraphs = []
    for line in lines:
        if paragraphs and should_join(paragraphs[-1], line):
            paragraphs[-1] += line
        else:
            paragraphs.append(line)

    metadata = dict(doc.metadata)
    metadata["source_name"] = Path(metadata.get("source", "")).name
    metadata["page_number"] = metadata.get("page", 0) + 1
    metadata["cleaned"] = True

    return Document(
        page_content="\n".join(paragraphs).strip(),
        metadata=metadata,
    )


# clean every page of the optimised version; keep the raw pages so the before/after comparison stays possible
cleaned_pdf_pages = [
    clean_pdf_document(page)
    for page in optimized_pdf_pages
    if page.page_content.strip()
]
cleaned_pdf_page = cleaned_pdf_pages[20]

# Before/after comparison and basic quality checks
print("===== before cleaning (first 500 characters) =====")
print(optimized_pdf_pages[20].page_content[:500])
print("\n===== after cleaning (first 500 characters) =====")
print(cleaned_pdf_page.page_content[:500])
print("\n===== quality statistics =====")
print("raw page count:", len(optimized_pdf_pages))
print("valid page count after cleaning:", len(cleaned_pdf_pages))
print("raw character count:", sum(len(page.page_content) for page in optimized_pdf_pages))
print("character count after cleaning:", sum(len(page.page_content) for page in cleaned_pdf_pages))
print("pages still carrying a private-area bullet:", sum("\uf06c" in page.page_content for page in cleaned_pdf_pages))
print("metadata of the current page:", cleaned_pdf_page.metadata)

### 2.3 Splitting

A single document is usually longer than the context a model supports, so retrieved knowledge would exceed the model's processing capacity. When building a vector knowledge base we therefore split documents into chunks, by length or by a fixed rule, embed each chunk, and store the vectors in a vector database.

At retrieval time the chunk is the atomic unit: each query returns k chunks as the knowledge the model may use to answer the user's question, and k is freely configurable.

Every LangChain text splitter splits according to `chunk_size` (chunk length) and `chunk_overlap` (overlap between neighbouring chunks).

![image.png](../../assets/rag-08.png)

* chunk_size is the number of characters or tokens (words, sentences, ...) in each chunk

* chunk_overlap is the number of characters two chunks share, used to keep context coherent so splitting does not lose information


**LangChain offers several splitters; they differ in how chunk boundaries are decided, what characters/tokens a chunk consists of, and how chunk size is measured**

- RecursiveCharacterTextSplitter(): splits text by characters, recursively trying different separators.
- CharacterTextSplitter(): splits text by characters.
- MarkdownHeaderTextSplitter(): splits a Markdown file based on the specified headings.
- TokenTextSplitter(): splits text by tokens.
- SentenceTransformersTokenTextSplitter(): splits text by tokens.
- Language(): for CPP, Python, Ruby, Markdown and similar.
- NLTKTextSplitter(): splits text into sentences using NLTK (Natural Language Toolkit).
- SpacyTextSplitter(): splits text into sentences using SpaCy.


In [ ]:
''' 
* RecursiveCharacterTextSplitter: recursive character text splitting
RecursiveCharacterTextSplitter splits recursively on different characters (in this priority order ["\n\n", "\n", " ", ""]),
    which keeps semantically related content together in the same chunk for as long as possible
RecursiveCharacterTextSplitter has 4 parameters that matter:

* separators - the list of separator strings
* chunk_size - the character limit of each chunk
* chunk_overlap - the length of the overlap between two neighbouring chunks
* length_function - the function used to measure length
'''
# import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# maximum length of a single chunk in the knowledge base
CHUNK_SIZE = 512

# overlap between two neighbouring chunks in the knowledge base
OVERLAP_SIZE = 50

In [ ]:
# the second stage only splits chapters that are too long; the chapter boundaries of the first stage are detected by the functions below
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "\u3002", "\uff1b", "\uff01", "\uff1f", "\uff0c", ""],
)

SECTION_TITLE_PATTERN = re.compile(
    "^(?:\u7b2c[\u4e00\u4e8c\u4e09\u56db\u4e94\u516d\u4e03\u516b\u4e5d\u5341\u767e]+\u7ae0(?:\\s+.+)?|\\d+(?:\\.\\d+)+(?:\\s+.+)|\\d+\\s+[^0-9].+)"
)
BUSINESS_TYPE_PATTERN = re.compile("\uff08([^\uff08\uff09]+\u4e1a\u52a1)\uff09")


def is_section_title(line: str) -> bool:
    # table-of-contents entries usually contain dot leaders and must not become chapter boundaries
    return not re.search("[.\uff0e\u00b7\u2026]{3,}", line) and bool(SECTION_TITLE_PATTERN.match(line))


def extract_business_type(source_name: str) -> str:
    match = BUSINESS_TYPE_PATTERN.search(source_name)
    return match.group(1) if match else "\u672a\u77e5\u4e1a\u52a1"


def build_section_documents(pages):
    """Detect chapters document by document and page by page; when a chapter spans pages, inherit the last section detected on the previous page."""
    section_docs = []
    current_section_by_source = {}

    for page in pages:
        metadata = dict(page.metadata)
        source_name = metadata.get("source_name") or Path(metadata.get("source", "")).name
        current_section = current_section_by_source.get(source_name, "\u672a\u8bc6\u522b\u7ae0\u8282")
        buffer = []

        def flush_buffer():
            if not buffer:
                return
            block_text = "\n".join(buffer).strip()
            # a page may end with the next section title while the body starts on the next page; avoid producing empty title-only chunks
            if block_text == current_section:
                buffer.clear()
                return
            block_metadata = dict(metadata)
            block_metadata.update({
                "source_name": source_name,
                "business_type": extract_business_type(source_name),
                "section_title": current_section,
            })
            section_docs.append(Document(
                page_content=block_text,
                metadata=block_metadata,
            ))
            buffer.clear()

        for line in page.page_content.splitlines():
            if is_section_title(line):
                flush_buffer()
                current_section = line
                current_section_by_source[source_name] = current_section
            buffer.append(line)
        flush_buffer()

    return section_docs


def split_documents_by_section(pages):
    """Split chapter-first with a length fallback, and give every source a stable chunk_index."""
    from collections import defaultdict

    section_docs = build_section_documents(pages)
    chunks = text_splitter.split_documents(section_docs)
    source_counters = defaultdict(int)

    for chunk in chunks:
        source_name = chunk.metadata["source_name"]
        chunk.metadata["chunk_index"] = source_counters[source_name]
        source_counters[source_name] += 1

        # the section title also goes into the embedding/BM25 text, which lifts recall for questions about section names
        section_title = chunk.metadata["section_title"]
        if section_title != "\u672a\u8bc6\u522b\u7ae0\u8282" and not chunk.page_content.startswith(section_title):
            chunk.page_content = f"{section_title}\n{chunk.page_content}"

    return chunks

In [ ]:
# split by section title first, then apply a second split to sections longer than CHUNK_SIZE
split_docs = split_documents_by_section(cleaned_pdf_pages)
print(f"number of chunks after section-aware splitting: {len(split_docs)}")
print("metadata example:", {
    key: split_docs[0].metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})

In [ ]:
split_docs[50].page_content

In [ ]:
required_metadata = {"source_name", "page_number", "business_type", "section_title", "chunk_index"}
print(f"character count after splitting: {sum(len(doc.page_content) for doc in split_docs)}")
print("is the metadata complete:", all(
    required_metadata.issubset(doc.metadata) for doc in split_docs
))
print("dot-leader lines mistaken for headings:", sum(
    bool(re.search("[.\uff0e\u00b7\u2026]{3,}", doc.metadata["section_title"]))
    for doc in split_docs
))

Note: document splitting usually sets the floor of retrieval quality. This section uses a two-stage strategy: first recover semantic boundaries from numbered chapter titles, then apply a second split to chapters that are too long using `chunk_size` and `chunk_overlap`. Dot leaders in the table of contents are excluded, and body text that spans pages inherits the chapter title from the previous page. Every chunk also stores source file, page number, business type, chapter title and in-document index, which makes filtered retrieval, citation tracing and issue localisation possible.


## 3. PDF-to-Markdown preprocessing

This part adds a third preprocessing route: `PDF -> Markdown -> split by heading -> second split for oversized sections`. It does not replace the optimised version; it is a quantified control experiment.

### 3.1 Layout-aware PDF-to-Markdown conversion

This maps to the "document loaders" and "structured parsing" sections of the foundations module C02 (data preparation), lesson 01 (data loading). Instead of extracting plain text only, it also uses font size, font name and page coordinates to restore headings and lists, and saves the page number explicitly.


In [ ]:
import fitz

MARKDOWN_PDF_PATH = (
    "../knowledge_path/VMAX-S/ZXVMAX-S\uff08V6.23\uff09\u4ea7\u54c1\u63cf\u8ff0\uff08\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1\uff09.pdf"
)

def _join_pdf_wrapped_lines(lines):
    """Merge the hard line breaks introduced by PDF layout, keeping natural paragraphs formed by sentence ends."""
    paragraphs = []
    buffer = ""
    for line in lines:
        buffer += line
        if line.endswith(("\u3002", "\uff01", "\uff1f", "\uff1b", "\uff1a")):
            paragraphs.append(buffer)
            buffer = ""
    if buffer:
        paragraphs.append(buffer)
    return paragraphs

def pdf_page_to_markdown(page, page_number):
    """Rebuild one PDF page into Markdown with page markers, using font size and position."""
    records = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            text = "".join(span["text"] for span in spans).replace("\u00a0", " " ).strip()
            if not text:
                continue
            y0 = line["bbox"][1]
            if y0 < 75 or y0 > page.rect.height - 80:
                continue
            records.append({
                "text": text,
                "size": max(span["size"] for span in spans),
                "font": " ".join(span["font"] for span in spans),
                "y": y0,
                "x": line["bbox"][0],
            })
    # bullets and text on the same visual line can differ in baseline by 1-2pt, so group by 4pt and then sort by x
    records.sort(key=lambda item: (round(item["y"] / 4), item["x"]))

    markdown_lines = [f"<!-- page_number: {page_number} -->", ""]
    body_lines = []
    pending_bullet = False

    def flush_body():
        if body_lines:
            markdown_lines.extend(_join_pdf_wrapped_lines(body_lines))
            markdown_lines.append("")
            body_lines.clear()

    for record in records:
        text = record["text"]
        if text in {"\uf06c", "•", "●", "▪"}:
            flush_body()
            pending_bullet = True
            continue
        if pending_bullet:
            flush_body()
            markdown_lines.extend([f"- **{text}**", ""])
            pending_bullet = False
            continue
        if record["size"] >= 15 and not re.search("[.\uff0e\u00b7\u2026]{3,}", text):
            flush_body()
            level = 2 if re.match(r"^\d+(?:\.\d+)+\s+", text) else 1
            markdown_lines.extend([f"{'#' * level} {text}", ""])
        elif (
            record["size"] >= 11
            and "ZHONGCU" in record["font"].upper()
            and not re.search("[.\uff0e\u00b7\u2026]{3,}", text)
        ):
            flush_body()
            markdown_lines.extend([f"### {text}", ""])
        else:
            body_lines.append(text)
    flush_body()
    return "\n".join(markdown_lines).strip()

def pdf_to_markdown_documents(pdf_path):
    pdf = fitz.open(pdf_path)
    source_name = Path(pdf_path).name
    docs = []
    for page_index, page in enumerate(pdf):
        docs.append(Document(
            page_content=pdf_page_to_markdown(page, page_index + 1),
            metadata={
                "source": pdf_path,
                "source_name": source_name,
                "page_number": page_index + 1,
                "business_type": extract_business_type(source_name),
                "content_format": "markdown",
            },
        ))
    return docs

markdown_pages = pdf_to_markdown_documents(MARKDOWN_PDF_PATH)
print(markdown_pages[20].page_content[:1500])

In [ ]:
from pathlib import Path


# ===============================
# save the Markdown
# ===============================

OUTPUT_DIR = Path("../knowledge_md/VMAX-S")

# create the output directory automatically
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


pdf_path = Path(MARKDOWN_PDF_PATH)

md_path = (
    OUTPUT_DIR /
    pdf_path.with_suffix(".md").name
)


with open(
    md_path,
    "w",
    encoding="utf-8"
) as f:

    for doc in markdown_pages:

        f.write(
            doc.page_content
        )

        f.write(
            "\n\n"
        )


print(
    "Markdown saved:"
)

print(
    md_path
)

### 3.2 Building structural boundaries from Markdown headings

**Foundations link:** the "structure-aware chunking" section of the foundations module C02 (data preparation), lesson 02 (text splitting). First split with `MarkdownHeaderTextSplitter` on `# / ## / ###`, then apply a second recursive split to sections over the length limit.


In [ ]:
from collections import defaultdict
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "chapter"),
        ("##", "section"),
        ("###", "subsection"),
    ],
    strip_headers=False,
)

markdown_section_docs = []
for page_doc in markdown_pages:
    for section_doc in markdown_header_splitter.split_text(page_doc.page_content):
        section_doc.metadata = {**page_doc.metadata, **section_doc.metadata}
        section_doc.metadata["section_title"] = (
            section_doc.metadata.get("subsection")
            or section_doc.metadata.get("section")
            or section_doc.metadata.get("chapter")
            or "\u672a\u8bc6\u522b\u7ae0\u8282"
        )
        markdown_section_docs.append(section_doc)

markdown_split_docs = text_splitter.split_documents(markdown_section_docs)
markdown_chunk_counters = defaultdict(int)
for chunk in markdown_split_docs:
    source_name = chunk.metadata["source_name"]
    chunk.metadata["chunk_index"] = markdown_chunk_counters[source_name]
    markdown_chunk_counters[source_name] += 1

print(f"Markdown page count: {len(markdown_pages)}")
print(f"section count from headings: {len(markdown_section_docs)}")
print(f"chunk count after the second split: {len(markdown_split_docs)}")

### 3.3 Checking the Markdown and metadata quality

**Foundations link:** the layered-evaluation idea in the foundations module C06 (RAG evaluation), lesson 01 (evaluation overview). Check the conversion quality before indexing, instead of debugging after an answer goes wrong.


In [ ]:
markdown_required_metadata = {
    "source_name", "page_number", "business_type",
    "section_title", "chunk_index", "content_format",
}
assert all(markdown_required_metadata.issubset(doc.metadata) for doc in markdown_split_docs)
assert not any("\uf06c" in doc.page_content for doc in markdown_split_docs)
assert any(doc.metadata["section_title"] == "7.3 \u6570\u636e\u53ef\u9760\u6027" for doc in markdown_split_docs)
assert not any(
    re.search("[.\uff0e\u00b7\u2026]{3,}", doc.metadata["section_title"])
    for doc in markdown_split_docs
)
print("Markdown preprocessing quality check passed")
print("section examples detected:", sorted({
    doc.metadata["section_title"] for doc in markdown_split_docs
    if doc.metadata["section_title"] != "\u672a\u8bc6\u522b\u7ae0\u8282"
})[:12])

## 4. Preprocessing with Unstructured document elements

### 4.1 Parsing and inspecting document element types

**Foundations link:** the "Unstructured document processing library" and "supported document element types" sections of the foundations module C02 (data preparation), lesson 01 (data loading). Unstructured offers a unified `Title / NarrativeText / ListItem / Table / Header / Footer` element model. Count the types first, then decide how to split.

The current Chinese DITA PDF marks many short text blocks as `Title` in `fast` mode, so this experiment does not use `chunk_by_title` directly: Unstructured handles element and page-number extraction, and the reliable numbered-heading rule decides the section boundaries.
>
> **Optional dependency**: `unstructured[pdf]` is large (it pulls in inference-related dependencies) and is not installed in the default environment of this project. When it is missing, the cells below skip themselves and the rest of the chapter still runs.


In [ ]:
import os
import tempfile
from collections import Counter

# avoid the read-only HOME cache directory; this must be set before importing the Unstructured PDF module
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "rag-mpl-cache"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "rag-numba-cache"))

try:
    from unstructured.partition.pdf import partition_pdf
except ImportError:
    partition_pdf = None

if partition_pdf is None:
    unstructured_elements = []
    print("Unstructured is not installed, skipping this section. To try it: pip install \"unstructured[pdf]\"")
else:
    unstructured_elements = partition_pdf(
        filename=MARKDOWN_PDF_PATH,
        strategy="fast",
        include_page_breaks=True,
        languages=["eng", "chi_sim"],
    )
    unstructured_type_counts = Counter(
        getattr(element, "category", type(element).__name__)
        for element in unstructured_elements
    )
    print(f"Unstructured element count: {len(unstructured_elements)}")
    print("element type statistics:", dict(unstructured_type_counts))

### 4.2 Element cleaning, section assembly and the second split

**Foundations link:** the "structure-aware chunking" section of the foundations module C02 (data preparation), lesson 02 (text splitting), plus Unstructured's element-level chunking idea. Keep the element type and page number, filter footers, standalone page numbers and page separators, then assemble content around reliable section titles.


In [ ]:
def unstructured_elements_to_section_docs(elements, source_path):
    source_name = Path(source_path).name
    section_docs = []
    current_section = "\u672a\u8bc6\u522b\u7ae0\u8282"
    current_page = None
    buffer = []
    element_types = set()

    def flush_buffer():
        if not buffer or current_page is None:
            return
        paragraphs = []
        for text in buffer:
            if paragraphs and should_join(paragraphs[-1], text):
                paragraphs[-1] += text
            else:
                paragraphs.append(text)
        section_docs.append(Document(
            page_content="\n".join(paragraphs),
            metadata={
                "source": source_path,
                "source_name": source_name,
                "page_number": current_page,
                "business_type": extract_business_type(source_name),
                "section_title": current_section,
                "content_format": "unstructured",
                "parser_strategy": "fast",
                "element_types": sorted(element_types),
            },
        ))
        buffer.clear()
        element_types.clear()

    for element in elements:
        category = getattr(element, "category", type(element).__name__)
        if category == "PageBreak":
            continue
        metadata = element.metadata.to_dict()
        page_number = metadata.get("page_number")
        if page_number is None:
            continue
        text = normalize_line(str(element))
        text = BULLET_PATTERN.sub("- ", text)
        if not text or FOOTER_PATTERN.match(text) or PAGE_NUMBER_PATTERN.match(text):
            continue
        if REPEATED_HEADER_PATTERN.match(text):
            continue
        if current_page is not None and page_number != current_page:
            flush_buffer()
        current_page = page_number
        if is_section_title(text):
            flush_buffer()
            current_section = text
        buffer.append(text)
        element_types.add(category)
    flush_buffer()
    return section_docs

if unstructured_elements:
    unstructured_section_docs = unstructured_elements_to_section_docs(
        unstructured_elements, MARKDOWN_PDF_PATH
    )
    unstructured_split_docs = text_splitter.split_documents(unstructured_section_docs)
    unstructured_counters = defaultdict(int)
    for chunk in unstructured_split_docs:
        source_name = chunk.metadata["source_name"]
        chunk.metadata["chunk_index"] = unstructured_counters[source_name]
        unstructured_counters[source_name] += 1

    print(f"Unstructured section block count: {len(unstructured_section_docs)}")
    print(f"chunk count after the second split: {len(unstructured_split_docs)}")
else:
    unstructured_section_docs = []
    unstructured_split_docs = []
    print("Unstructured is not installed, skipping element cleaning and the second split.")

### 4.3 Checking the Unstructured result

This maps to the foundations module C06 (RAG evaluation), lesson 01 (evaluation overview). Check element types, sections, page numbers and leftover noise, and record the parser strategy in the metadata so different loaders can be compared later.


In [ ]:
if unstructured_split_docs:
    unstructured_required_metadata = {
        "source_name", "page_number", "business_type", "section_title",
        "chunk_index", "content_format", "parser_strategy", "element_types",
    }
    assert all(
        unstructured_required_metadata.issubset(doc.metadata)
        for doc in unstructured_split_docs
    )
    assert not any("\uf06c" in doc.page_content for doc in unstructured_split_docs)
    assert any(
        doc.metadata["section_title"] == "7.3 \u6570\u636e\u53ef\u9760\u6027"
        for doc in unstructured_split_docs
    )
    print("Unstructured preprocessing quality check passed")
else:
    print("Unstructured is not installed, skipping the quality check.")

## 5. Comparing the four preprocessing routes

The comparison below always uses the same block: page 21 of the same document, the block containing "7.3 data reliability". The simple version slices by length, the optimised version restores sections with regexes, the Markdown version restores structure from the PDF layout, and the Unstructured version uses a unified document-element model that keeps element types.


In [ ]:
COMPARE_PAGE_NUMBER = 21
COMPARE_SECTION = "7.3 \u6570\u636e\u53ef\u9760\u6027"

simple_example = next(
    doc for doc in simple_split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and "\u6570\u636e\u53ef\u9760\u6027" in doc.page_content
)
optimized_example = next(
    doc for doc in split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and doc.metadata.get("section_title") == COMPARE_SECTION
)
markdown_example = next(
    doc for doc in markdown_split_docs
    if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
    and doc.metadata.get("section_title") == COMPARE_SECTION
)
unstructured_example = None
if unstructured_split_docs:
    unstructured_example = next(
        (doc for doc in unstructured_split_docs
         if doc.metadata.get("page_number") == COMPARE_PAGE_NUMBER
         and doc.metadata.get("section_title") == COMPARE_SECTION),
        None,
    )

print("===== simple preprocessing chunk =====")
print(simple_example.page_content)
print("metadata:", simple_example.metadata)
print("\n===== optimised preprocessing chunk =====")
print(optimized_example.page_content)
print("metadata:", {
    key: optimized_example.metadata.get(key)
    for key in ["source_name", "page_number", "business_type", "section_title", "chunk_index"]
})
if unstructured_example is not None:
    print("\n===== Unstructured chunk =====")
    print(unstructured_example.page_content)
    print("metadata:", {
        key: unstructured_example.metadata.get(key)
        for key in [
            "source_name", "page_number", "business_type", "section_title",
            "chunk_index", "content_format", "parser_strategy", "element_types",
        ]
    })
print("\n===== PDF-to-Markdown chunk =====")
print(markdown_example.page_content)
print("metadata:", {
    key: markdown_example.metadata.get(key)
    for key in [
        "source_name", "page_number", "business_type",
        "section_title", "chunk_index", "content_format",
    ]
})
print("\n===== comparison conclusion =====")
print("character count, simple version:", len(simple_example.page_content))
print("character count, optimised version:", len(optimized_example.page_content))
print("character count, Markdown version:", len(markdown_example.page_content))
print("section title, optimised version:", optimized_example.metadata["section_title"])
print("section title, Markdown version:", markdown_example.metadata["section_title"])
print("Markdown version kept the list:", "- **\u91c7\u96c6\u5c42**" in markdown_example.page_content)
if unstructured_example is not None:
    print("character count, Unstructured version:", len(unstructured_example.page_content))
    print("Unstructured element types:", unstructured_example.metadata["element_types"])
else:
    print("Unstructured is not installed, this section only compares the other three preprocessing routes.")

## 6. Conclusion and selection advice

Given the actual output on the current ZXVMAX-S product manual, the four options rank:

1. **optimised preprocessing**: the current default for indexing;
2. **PDF-to-Markdown**: prefer it when headings, lists and table structure are prominent;
3. **Unstructured**: best when PDF, Word, HTML and Excel must be handled uniformly;
4. **simple preprocessing**: keep it as the teaching baseline only, never for real indexing.

| Approach | Main strengths | Main problems | Suggested use |
|---|---|---|---|
| simple preprocessing | easy to implement, fast | several topics mixed per chunk, hard line breaks and private symbols remain, metadata incomplete | teaching baseline |
| optimised preprocessing | high section purity, compact text, fast, few dependencies, controllable rules | the rules target this document format and need adapting for new formats | current default for indexing |
| PDF to Markdown | best at expressing heading levels, lists and tables | the conversion depends on fonts and layout, Markdown markers add a little text | strongly structured PDFs, documents heavy in tables and lists |
| Unstructured | unified element model, rich metadata, many file formats | heavier dependencies, slower, and `fast` mode over-classifies Title on this Chinese PDF | multi-format enterprise knowledge bases |

On the current sample the optimised and Unstructured bodies are of similar quality, but the optimised chunk holds only 360 characters, so it is denser and free of Unstructured misclassification, which makes it the better default for this project. The Markdown version keeps `##` headings and bolded list items and expresses structure best, so it is a good extra route for complex layouts.

The recommended strategy is to route by document type:

```text
DITA product manuals, text-only PDFs -> optimised preprocessing
PDFs with prominent headings, lists and tables -> PDF to Markdown
mixed PDF / Word / HTML / Excel knowledge bases -> Unstructured
simple preprocessing -> baseline comparison only
```

> This conclusion comes from comparing the quality of one block on one page of one section. Before replacing the default for real, compare Hit@K, MRR, section hit rate, page-number accuracy and answer faithfulness on the same evaluation set.


## 7. Batch preprocessing


Process every file in the folder in one pass


In [ ]:
import os

# collect every file path under folder_path into file_paths
file_paths = []
folder_path = '../knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# walk the file paths and keep the instantiated loaders in loaders
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")


# download the files and store them into text
# load every document into texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # key step: initialise texts


# reuse the cleaning functions defined earlier so the single-document and batch pipelines never diverge
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"raw page count: {len(raw_texts)}, valid page count after cleaning: {len(texts)}")

In [ ]:
''' 
* RecursiveCharacterTextSplitter: recursive character text splitting
RecursiveCharacterTextSplitter splits recursively on different characters (in this priority order ["\n\n", "\n", " ", ""]),
    which keeps semantically related content together in the same chunk for as long as possible
RecursiveCharacterTextSplitter has 4 parameters that matter:

* separators - the list of separator strings
* chunk_size - the character limit of each chunk
* chunk_overlap - the length of the overlap between two neighbouring chunks
* length_function - the function used to measure length
'''
# import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# maximum length of a single chunk in the knowledge base
CHUNK_SIZE = 512

# overlap between two neighbouring chunks in the knowledge base
OVERLAP_SIZE = 50

In [ ]:
# use the recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "\u3002", "\uff1b", "\uff01", "\uff1f", "\uff0c", ""],
)
# text_splitter.split_text(text.page_content[0:1000])

In [ ]:
split_docs = split_documents_by_section(texts)
print(f"number of chunks after splitting: {len(split_docs)}")

In [ ]:
print(f"character count after splitting (a rough proxy for the token count): {sum([len(doc.page_content) for doc in split_docs])}")

In [ ]:
split_docs[19].page_content

# Building the vector database


## 1. Installing Milvus


Milvus Standalone is the single-server deployment. All Milvus Standalone components are packaged into one [Docker image](https://milvus.io/docs/install_standalone-docker.md), which makes deployment very convenient. If you have production workloads but do not want Kubernetes, running Milvus Standalone on a single server with enough memory is a good choice. Milvus Standalone also supports high availability through primary-replica replication.

**Running Milvus with Docker Compose (Linux)**

The Milvus repository ships a Docker Compose configuration file. To install Milvus with Docker Compose, run

```shell
# Download the configuration file
$ wget https://github.com/milvus-io/milvus/releases/download/v3.0.3/milvus-standalone-docker-compose.yaml -O docker-compose.yaml


# Start Milvus
$ sudo docker compose -f docker-compose.yaml up -d

Creating milvus-etcd  ... done
Creating milvus-minio ... done
Creating milvus-standalone ... done
```

- If the command above fails, check whether the system has Docker Compose V1 installed. If so, migrate to Docker Compose V2 following [this page](https://docs.docker.com/compose/).
- If you hit problems pulling the images, contact us at [community@zilliz.com](mailto:community@zilliz.com) with the details and we will help.

After Milvus starts,

- the containers milvus-standalone, milvus-minio and milvus-etcd are running.
  - The **milvus-etcd** container exposes no ports to the host and maps its data to **volumes/etcd** in the current folder.
  - The **milvus-minio** container serves local ports **9090** and **9091** with the default credentials and maps its data to **volumes/minio** in the current folder.
  - The **milvus-standalone** container serves local port **19530** with default settings and maps its data to **volumes/milvus** in the current folder.

Check that the containers are up with:

```shell
$ sudo docker-compose ps

      Name                     Command                  State                            Ports
--------------------------------------------------------------------------------------------------------------------
milvus-etcd         etcd -advertise-client-url ...   Up             2379/tcp, 2380/tcp
milvus-minio        /usr/bin/docker-entrypoint ...   Up (healthy)   9000/tcp
milvus-standalone   /tini -- milvus run standalone   Up             0.0.0.0:19530->19530/tcp, 0.0.0.0:9091->9091/tcp
```

You can also open the Milvus WebUI at `http://127.0.0.1:9091/webui/` to learn more about the Milvus instance. See [Milvus WebUI](https://milvus.io/docs/milvus-webui.md) for details.

**Stopping and removing Milvus**

Stop and remove the containers as follows

```shell
# Stop Milvus
$ sudo docker compose down

# Delete service data
$ sudo rm -rf volumes
```


**Attu (the Milvus GUI)**

https://github.com/zilliztech/attu

```shell
# Attu v3 points at Milvus through MILVUS_ADDRESS; use host.docker.internal to reach the host from inside the container
docker run -d --name attu-container \
  -p 9092:3000 \
  -e MILVUS_ADDRESS=http://host.docker.internal:19530 \
  -v attu-data:/data \
  zilliz/attu:v3.0.0
```

After it starts, open `http://localhost:9092` and enter the same Milvus address on the login page - no username or password is needed.

> Never write the login account or password into the notebook; if Attu authentication is enabled, configure it through deployment environment variables or a password manager.

**Milvus user guide**

- [Databases](https://milvus.io/docs/manage_databases.md)
- Collections
- Schema and data fields
- Insert and delete
- Indexes
- Search and rerankers

> See https://milvus.io/docs/manage_databases.md


## 2. Upstream data preparation


In [ ]:
#!/usr/bin/env python
# coding: utf-8

# process every file in the folder in one pass
import os

# collect every file path under folder_path into file_paths
file_paths = []
folder_path = '../knowledge_path/VMAX-S'
for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        file_paths.append(file_path)
print(file_paths)

from langchain_community.document_loaders import PyMuPDFLoader
# from langchain_community.document_loaders import UnstructuredMarkdownLoader

# walk the file paths and keep the instantiated loaders in loaders
loaders = []

for file_path in file_paths:
    file_type = file_path.split('.')[-1]
    if file_type == 'pdf':
        loaders.append(PyMuPDFLoader(file_path))
    else:
        print(f"Unsupported file type: {file_type} for file {file_path}")

# download the files and store them into text
# load every document into texts
texts = []
for loader in loaders:
    texts.extend(loader.load())  # key step: initialise texts


# reuse the cleaning functions defined earlier
raw_texts = texts
texts = [clean_pdf_document(doc) for doc in raw_texts if doc.page_content.strip()]
print(f"raw page count: {len(raw_texts)}, valid page count after cleaning: {len(texts)}")




# import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
''' 
* RecursiveCharacterTextSplitter: recursive character text splitting
RecursiveCharacterTextSplitter splits recursively on different characters (in this priority order ["\n\n", "\n", " ", ""]),
    which keeps semantically related content together in the same chunk for as long as possible
RecursiveCharacterTextSplitter has 4 parameters that matter:

* separators - the list of separator strings
* chunk_size - the character limit of each chunk
* chunk_overlap - the length of the overlap between two neighbouring chunks
* length_function - the function used to measure length
'''


# maximum length of a single chunk in the knowledge base
CHUNK_SIZE = 512

# overlap between two neighbouring chunks in the knowledge base
OVERLAP_SIZE = 50


# use the recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=OVERLAP_SIZE,
    separators=["\n\n", "\n", "\u3002", "\uff1b", "\uff01", "\uff1f", "\uff0c", ""],
)


split_docs = split_documents_by_section(texts)
print(f"number of chunks after splitting: {len(split_docs)}")



print(f"character count after splitting (a rough proxy for the token count): {sum([len(doc.page_content) for doc in split_docs])}")



split_docs[100].page_content



In [ ]:
#!/usr/bin/env python
# coding: utf-8


import os
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter



# =====================================
# directory configuration
# =====================================

INPUT_DIR = Path("../knowledge_path/VMAX-S")

OUTPUT_DIR = Path("../knowledge_md/VMAX-S")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# =====================================
# collect the PDF files
# =====================================

file_paths = []

for root, dirs, files in os.walk(INPUT_DIR):

    for file in files:

        if file.lower().endswith(".pdf"):

            file_paths.append(
                Path(root) / file
            )


print("PDF file count:", len(file_paths))



# =====================================
# load the PDFs
# =====================================

all_documents = []


for pdf_file in file_paths:

    print("\nloading:", pdf_file.name)


    loader = PyMuPDFLoader(
        str(pdf_file)
    )


    docs = loader.load()


    # attach the source information
    for doc in docs:

        doc.metadata["source_file"] = pdf_file.name


    all_documents.extend(docs)



print(
    "PDF page count:",
    len(all_documents)
)



# =====================================
# clean the text
# =====================================

clean_docs = []


for doc in all_documents:


    text = doc.page_content


    if not text.strip():
        continue


    # remove empty lines
    text = "\n".join(
        line.strip()
        for line in text.splitlines()
        if line.strip()
    )


    doc.page_content = text


    clean_docs.append(doc)



print(
    "pages after cleaning:",
    len(clean_docs)
)



# =====================================
# split the text
# =====================================

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=100,


    separators=[

        "\n\n",

        "\n",

        "\u3002",

        "\uff01",

        "\uff1f",

        "\uff1b",

        "\uff0c",

        ""

    ]
)



split_docs = text_splitter.split_documents(
    clean_docs
)



print(
    "chunk count:",
    len(split_docs)
)



# =====================================
# save one Markdown file per PDF name
# =====================================

from collections import defaultdict


pdf_groups = defaultdict(list)



for doc in split_docs:

    filename = doc.metadata["source_file"]

    pdf_groups[filename].append(doc)



# one PDF produces one Markdown file

for pdf_name, docs in pdf_groups.items():


    md_name = Path(pdf_name).stem + ".md"


    md_path = OUTPUT_DIR / md_name


    print(
        "generated:",
        md_path
    )


    with open(
        md_path,
        "w",
        encoding="utf-8"
    ) as f:


        f.write(
            f"# {Path(pdf_name).stem}\n\n"
        )


        for i, doc in enumerate(docs):


            f.write(
                f"## Chunk {i+1}\n\n"
            )


            f.write(
                doc.page_content
            )


            f.write(
                "\n\n"
            )



print("\n===== all done =====")

## 3. Preparing the embedding model


In [ ]:
import sys
print(sys.executable)

import langchain_core
print(langchain_core.__version__)

import langchain_core.messages as m
print(m.__file__)

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from langchain_openai import OpenAIEmbeddings

# initialise the embedding model
my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)


## 4. Building the Milvus vector store: simple retrieval version


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus

# Milvus connection parameters. The Quickstart connects with MilvusClient(uri=...); here we connect to a remote Milvus Standalone.
connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# the simple retrieval collection holds a single dense vector field, i.e. the single-vector semantic search from the Milvus Quickstart.
simple_collection_name = "ZXVMAXS"

# Dense: HNSW + COSINE -- similarity search over vector semantics.
dense_index_param = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {
        "M": 16,
        "efConstruction": 200,
    },
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"processing batch {batch_num + 1}/{total_batches} (documents {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                collection_name=simple_collection_name,
                connection_args=connection_args,
                vector_field="dense",
                text_field="text",
                index_params=dense_index_param,
                drop_old=True,
            )
        else:
            vectordb.add_documents(batch_docs)

        print(f"batch {batch_num + 1} imported into the Milvus simple retrieval collection")

    # at search time we only need the text and metadata fields, never the raw dense vector.
    vectordb.fields = [field for field in vectordb.fields if field not in {"dense", "sparse", "vector"}]
    print(f"every document was imported into the simple retrieval collection: {simple_collection_name}")
    print(f"vector fields: {vectordb.vector_fields}")
    print(f"returned fields: {vectordb.fields}")

except Exception as e:
    print(f"error while building the Milvus simple retrieval collection: {e}")


## 5. Building the Milvus vector store: hybrid retrieval version


Follow the basic flow in the [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md): connect to Milvus, create a collection, write vector and scalar fields, and return only the text and metadata fields you want to display when searching. With the LangChain wrapper this maps to:

- `dense`: produced by the embedding model and used for semantic similarity retrieval;
- `sparse`: produced on the Milvus server side by the built-in BM25 function from the original text, used for keyword retrieval;
- at search time the dense and sparse recall results are fused with `ranker_type="weighted"` or `ranker_type="rrf"`;
- `sparse` is the sparse vector field output by the BM25 function and cannot be returned as raw data, so exclude `dense/sparse/vector` from `fields` before searching.

`BM25BuiltInFunction` requires no corpus statistics on the client: text analysis, sparse vector generation, and BM25 scoring all happen on the Milvus server side. Note: Milvus Lite does not yet support this built-in full-text search capability - use Milvus Standalone, Distributed, or a compatible service.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# the hybrid collection uses its own name, to avoid a schema clash with the single-vector simple collection.
hybrid_collection_name = "ZXVMAXS_HYBRID"

# Dense: HNSW + COSINE -- similarity search over vector semantics.
# Sparse: AUTOINDEX + BM25 -- retrieval by keyword relevance.
# Hybrid Search -- fuse the two result sets, combining semantic understanding with keyword matching.
dense_index_param = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {
        "M": 16,
        "efConstruction": 200,
    },
}
sparse_index_param = {
    "metric_type": "BM25",
    "index_type": "AUTOINDEX",
}

batch_size = 30

try:
    total_batches = (len(split_docs) + batch_size - 1) // batch_size
    vectordb_hybrid = None

    for batch_num in range(total_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(split_docs))
        batch_docs = split_docs[start_idx:end_idx]

        print(f"processing batch {batch_num + 1}/{total_batches} (documents {start_idx}-{end_idx - 1})")

        if batch_num == 0:
            vectordb_hybrid = Milvus.from_documents(
                documents=batch_docs,
                embedding=my_emb,
                builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
                vector_field=["dense", "sparse"],
                collection_name=hybrid_collection_name,
                connection_args=connection_args,
                consistency_level="Bounded",
                index_params=[dense_index_param, sparse_index_param],
                drop_old=True,
            )
        else:
            vectordb_hybrid.add_documents(batch_docs)

        print(f"batch {batch_num + 1} imported into the Milvus hybrid retrieval collection")

    # search results only return text and metadata fields; Milvus refuses to return the sparse raw data produced by the BM25 function.
    vectordb_hybrid.fields = [
        field for field in vectordb_hybrid.fields
        if field not in {"dense", "sparse", "vector"}
    ]
    print(f"every document was imported into the hybrid retrieval collection: {hybrid_collection_name}")
    print(f"vector fields: {vectordb_hybrid.vector_fields}")
    print(f"returned fields: {vectordb_hybrid.fields}")

except Exception as e:
    print(f"error while building the Milvus hybrid retrieval collection: {e}")


Below we run one quick retrieval against the same Milvus collection. The order of `weights` matches `vector_field=["dense", "sparse"]`: the first weight controls semantic vector retrieval, the second controls BM25 keyword retrieval.


In [ ]:
query = "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f"

# Milvus will not return sparse/dense vector fields as raw data; keep only ordinary fields before the hybrid search.
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

hybrid_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

for i, doc in enumerate(hybrid_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
    page = doc.metadata.get("page", "-")
    print(f"===== hybrid retrieval result {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


In [ ]:
# You can also fuse the two recall sets with RRF (Reciprocal Rank Fusion), which suits cases where you do not want to tune weights by hand.
# Milvus will not return sparse/dense vector fields as raw data; keep only ordinary fields before the hybrid search.
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

for i, doc in enumerate(rrf_docs, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
    page = doc.metadata.get("page", "-")
    print(f"===== RRF hybrid retrieval result {i} | source={source} | page={page} =====")
    print(doc.page_content[:500])
    print()


## 6. Connecting to the database for testing


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from pymilvus import MilvusClient

_ = load_dotenv(find_dotenv())

def list_all_collections(uri: str):
    """List every collection."""
    try:
        client = MilvusClient(uri=uri)
        collections = client.list_collections()
        print("collection list:")
        for i, coll_name in enumerate(collections, 1):
            print(f"{i}. {coll_name}")
        return collections
    except Exception as e:
        print("failed to list collections:", e)
        return []

# run it
collections = list_all_collections(os.environ["MILVUS_URI"])
print(f"found {len(collections)} collections in total")

## 7. Comparing simple retrieval and hybrid retrieval results

Following the retrieval flow in the [Milvus Quickstart](https://milvus.io/docs/v2.6.x/quickstart.md): first write document vectors and metadata into a collection, then issue a `search` with the same query and inspect the Top-K matched text and metadata. We use the same idea here to compare the two Milvus collections created above:

- **Simple retrieval version `ZXVMAXS`**: uses only the `dense` vector field and relies mainly on semantic similarity recall;
- **Hybrid retrieval version `ZXVMAXS_HYBRID`**: uses both the `dense` semantic vectors and the `sparse` BM25 keyword vectors, then fuses the results with a weighted ranker.

When the query contains keywords such as product names, alarm codes, or feature names, hybrid retrieval usually ranks the exact keyword matches higher; when the query is closer to natural-language phrasing, simple vector retrieval and hybrid retrieval can overlap quite a lot.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",  # simple retrieval collection: dense vector field only
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",  # hybrid collection: two vector fields, dense + sparse
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# Milvus will not return vector fields as raw data, in particular the sparse field produced by BM25.
# deleting dense/sparse/vector explicitly is more robust than relying on vector_fields alone.
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]


In [ ]:
comparison_query = "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f"
k = 5

# Milvus will not return sparse/dense vector fields as raw data; keep only ordinary fields before searching.
for vectorstore in [vectordb, vectordb_hybrid]:
    vectorstore.fields = [
        field for field in vectorstore.fields
        if field not in {"dense", "sparse", "vector"}
    ]

print("fields returned by simple retrieval:", vectordb.fields)
print("fields returned by hybrid retrieval:", vectordb_hybrid.fields)

simple_docs = vectordb.similarity_search(comparison_query, k=k)
hybrid_docs = vectordb_hybrid.similarity_search(
    comparison_query,
    k=k,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},  # dense weight 0.6, BM25 sparse weight 0.4
)

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page

def print_results(title, docs):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        display_key = (source, page)
        if display_key in displayed:
            continue
        displayed.add(display_key)
        preview = doc.page_content[:180].replace("\n", " ")
        print(f"{row}. source={source} | page={page}")
        print(f"   {preview}")
        row += 1

print(f"Query: {comparison_query}")
print_results("simple retrieval Top-K (dense semantic search)", simple_docs)
print_results("hybrid retrieval Top-K (dense + BM25 sparse weighted fusion)", hybrid_docs)

simple_keys = {doc_key(doc): i + 1 for i, doc in enumerate(simple_docs)}
hybrid_keys = {doc_key(doc): i + 1 for i, doc in enumerate(hybrid_docs)}
common_keys = set(simple_keys) & set(hybrid_keys)
only_simple = set(simple_keys) - set(hybrid_keys)
only_hybrid = set(hybrid_keys) - set(simple_keys)

print("\n===== Top-K difference statistics =====")
print(f"shared recalls: {len(common_keys)}/{k}")
print(f"hit only by simple retrieval: {len(only_simple)}")
print(f"hit only by hybrid retrieval: {len(only_hybrid)}")

if only_hybrid:
    print("\nexamples only found by hybrid retrieval:")
    hybrid_doc_by_key = {doc_key(doc): doc for doc in hybrid_docs}
    for key in list(only_hybrid)[:3]:
        doc = hybrid_doc_by_key[key]
        source, page = key
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"- source={source} | page={page} | {preview}")

print("\nconclusion:")
print("- simple retrieval only reads dense vector similarity, so it suits questions that were heavily paraphrased.")
print("- hybrid retrieval fuses dense semantic recall with BM25 keyword recall, so it suits questions containing product names, module names, alarm codes and proper nouns.")
print("- a low shared-recall count means the keyword channel visibly changed the ranking or the candidate set; a high count means semantic recall is already stable for this query.")


### 7.1 Reading the comparison

In this real environment run, `ZXVMAXS` and `ZXVMAXS_HYBRID` were both built from the same `split_docs` batch, so a high overlap between their Top-K results is normal. The query about ZXVMAX-S internet-surfing log features (the Chinese query used in the code above) contains both a product name and a feature-type keyword, and `dense` semantic retrieval already recalls the "internet-surfing log service" pages reliably, so simple retrieval and hybrid retrieval hit 4 documents in common.

Compared with simple retrieval, hybrid retrieval additionally hit the feature-overview page inside the "end-to-end service" document, which shows that the BM25 keyword channel reinforces literal matches such as `ZXVMAX-S` and the word for "feature", and then fuses them with the dense semantic recall for ranking. This result illustrates the difference between the two:

- simple retrieval relies more on semantic similarity and suits questions rewritten in natural language;
- hybrid retrieval considers both semantic similarity and keyword hits, and suits questions containing proper terms such as product names, module names, alarm codes, and interface names;
- the search only returns ordinary fields such as `text/source/page` and not the `dense/sparse` vector fields, which matches the Milvus Quickstart practice of controlling returned fields through `output_fields` and avoids the `not allowed to retrieve raw data of field sparse` error.


# RAG basics: building a basic retrieval question-answering chain


This chapter builds the RAG question-answering chain on top of the **simple retrieval vector store `ZXVMAXS`** created in the previous chapter. The flow breaks into four steps:

1. connect to Milvus and load the simple retrieval collection that contains only the `dense` vector field;
2. turn the user question into a vector and recall the Top-K relevant document chunks from Milvus;
3. splice the recalled chunks into the prompt, constraining the LLM to answer only from the context;
4. compare the answers "with" and "without" the knowledge base to observe how RAG affects domain questions and hallucination suppression.

This chapter deliberately does not use BM25 sparse hybrid retrieval, so that the most basic dense vector retrieval chain can be taught clearly first.


## 1. Loading the Milvus vector database

Here we load the simple retrieval collection `ZXVMAXS` created in the previous chapter. Three things need attention:

- `embedding_function` must match the embedding model used when the vector store was built;
- the collection's vector field is named `dense` and its text field is named `text`, so LangChain's default `vector` cannot be used;
- the retrieval results only need to return the text and metadata fields, not the raw vector fields, so `dense/sparse/vector` is later excluded from `fields`.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)

# search results only need the text and metadata fields, never the raw dense vector.
vectordb.fields = [
    field for field in vectordb.fields
    if field not in {"dense", "sparse", "vector"}
]

print("vector fields:", vectordb.vector_fields)
print("returned fields:", vectordb.fields)


In [ ]:
results = vectordb.similarity_search(query="\u4ec0\u4e48\u662fvmax\u7684\u4e0a\u7f51\u65e5\u5fd7\u7cfb\u7edf\uff1f", k=5)

for i, doc in enumerate(results, 1):
    source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
    page = doc.metadata.get("page", "-")
    preview = doc.page_content[:220].replace("\n", " ")
    print(f"===== recalled result {i} | source={source} | page={page} =====")
    print(preview)
    print()


## 2. Creating the large language model


Here we initialise a chat model against Qwen3 deployed on vLLM (OpenAI-compatible interface). For teaching, think of it as the "generator": the retriever finds context in the knowledge base, the LLM organises the final answer from that context.

Before running, confirm that `vLLM_QWEN_BASE_URL`, `vLLM_QWEN_API_KEY` and `vLLM_QWEN_MODEL_ID` are configured in the repository root `.env`.


**vLLM version**


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

# -- 1. Chat model configuration ----------------------------------------
_ = load_dotenv(find_dotenv())

my_llm = init_chat_model(
    model=os.getenv("vLLM_QWEN_MODEL_ID"),
    model_provider="openai",
    base_url=os.getenv("vLLM_QWEN_BASE_URL"),
    api_key=os.getenv("vLLM_QWEN_API_KEY"),
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=10000,
)

**Alternative: switch to a cloud OpenAI-compatible API**


In [ ]:
# alternative: point the chat model at any cloud OpenAI-compatible service (placeholder, never commit a real key)
# import os
# import httpx
# from dotenv import load_dotenv, find_dotenv
# from langchain.chat_models import init_chat_model
#
# _ = load_dotenv(find_dotenv())
# cloud_api_key = os.getenv("CLOUD_LLM_API_KEY")   # configure it yourself in .env
# if not cloud_api_key:
#     raise ValueError("CLOUD_LLM_API_KEY is not configured; set it in the repository root .env first.")
#
# my_llm = init_chat_model(
#     model="your-cloud-model-name",
#     model_provider="openai",
#     base_url="https://your-endpoint/v1",
#     api_key=cloud_api_key,
#     http_client=httpx.Client(trust_env=False),
#     temperature=0.3,
#     max_tokens=3000,
# )
# print(my_llm.invoke("Who are you?").content)

## 3. Building the retrieval question-answering chain


The prompt is the key control point of the RAG question-answering chain. This template emphasises three things:

- answer only from the retrieved context, to reduce improvisation outside the knowledge base;
- if the context has no answer, answer explicitly "I don't know";
- keep the output concise and coherent, suitable for a teaching demo.


Build the system prompt


In [ ]:
from langchain_core.prompts import PromptTemplate

template = """You are the VMAX operations assistant. Answer strictly from the context below.

Context:
{context}

Question: {question}

Answer requirements:
1. Answer only from the context; never invent information that is not in it;
2. If the context does not contain the answer, reply "I don't know";
3. Keep the answer concise and coherent, without repeating the same information;
4. End with "Thanks for your question!".

Answer:"""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)


Create a template-based LCEL retrieval chain. LCEL (LangChain Expression Language) lets you wire "retrieve documents -> format the context -> fill the prompt -> call the LLM -> parse the output" into one composable chain.


In [ ]:
from langchain_classic.chains import RetrievalQA

# the baseline keeps plain similarity search so the flow stays easy to follow: question vector -> Milvus Top-K chunks.
# for more diverse results, set search_type to "mmr" and configure fetch_k/lambda_mult.
base_retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)


In [ ]:
def format_docs(docs):
    """Assemble the recalled documents into the prompt context, keeping simple source markers so the evidence stays visible."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "\u672a\u77e5\u6765\u6e90")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "\u672a\u8bc6\u522b\u7ae0\u8282")
        formatted.append(
            f"[doc {i} | source={source} | page={page} | section={section}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

answer_chain = (
    {
        "context": lambda x: format_docs(x["source_documents"]),
        "question": lambda x: x["question"],
    }
    | QA_CHAIN_PROMPT
    | my_llm
    | StrOutputParser()
)

# keep the raw recalled documents first, then generate the answer from them.
# finally return question, source_documents and result in one shape so the answer's provenance stays visible.
rag_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        source_documents=base_retriever,
    )
    | RunnablePassthrough.assign(result=answer_chain)
)

In [ ]:
question_1 = "\u4ec0\u4e48\u662fZXVMAX-S\u76845GC\u4e1a\u52a1\uff1f"
result = rag_chain.invoke(question_1)
print("LCEL retrieval chain answer for question_1:")
print(result["result"])

## 4. Testing the retrieval question-answering chain

We test the basic RAG chain with two questions:

- `question_1` is a domain question inside the knowledge base, used to see whether RAG can recall evidence from the VMAX documents and generate an answer;
- `question_2` is a general-knowledge question outside the knowledge base, used to see whether the prompt constrains the model to answer "I don't know" when the context is missing.

> **How to read the output**: the prompt in this project explicitly requires "answer only 'I don't know' when the context is insufficient", and Qwen3 on vLLM is quite conservative. When the top retrieved chunks are table-of-contents or chart pages with low information density, the RAG chain simply refuses, while the no-knowledge-base baseline writes more. This is not a pipeline failure, it is the combined result of **retrieval quality plus a strict constraint-style prompt**. When teaching, first check the printed source file names and page numbers to judge whether the answer actually hit the body text, then look at the evaluation step: the advantage of RAG shows up in **faithfulness (not fabricating)**, not in answer length. For a more forthcoming result, increase `k`, switch to hybrid retrieval, or relax the refusal constraint in the prompt.


### 4.1 Question inside the knowledge base: answering a domain question

This question goes through the complete RAG flow of "vectorise the question -> Milvus similarity search -> fill the retrieved chunks into the prompt -> LLM generates the answer". The code also prints the first 3 source documents so learners can see where the answer is grounded.


In [ ]:
question_1 = "\u4ec0\u4e48\u662fZXVMAX-S\u76845GC\u4e1a\u52a1\uff1f"
result = rag_chain.invoke(question_1)

print("LLM + knowledge base answer for question_1:")
print(result["result"])
print("source document count:", len(result["source_documents"]))
for i, doc in enumerate(result["source_documents"][:3], 1):
    source = (doc.metadata.get("source_name") or doc.metadata.get("source")
              or doc.metadata.get("file_path") or "\u672a\u77e5\u6765\u6e90")
    page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
    section = doc.metadata.get("section_title", "\u672a\u8bc6\u522b\u7ae0\u8282")
    print(f"source {i}: source={source} | page={page} | section={section}")


### 4.2 Question outside the knowledge base: refusal test

This question is unrelated to the VMAX knowledge base. Ideally the retrieved context cannot support an answer, so the prompt constrains the model to output "I don't know" instead of improvising from general knowledge.


In [ ]:
question_2 = "Who is Jay Chou?"
result = rag_chain.invoke(question_2)

print("LLM + knowledge base answer for question_2:")
print(result["result"])


### 4.3 Comparison with an LLM answer that has no knowledge base

To show what RAG contributes, the following answers the same questions directly with the LLM, bypassing the vector database. When comparing, focus on:

- whether the domain question tends to produce generic, inaccurate, or off-document answers;
- whether the out-of-knowledge-base question gets answered directly instead of refused.


In [ ]:
prompt_template = f"Please answer the following question:\n{question_1}"

raw_result_1 = my_llm.invoke(prompt_template)
print("answer for question_1 without a knowledge base:")
print(raw_result_1.content)


In [ ]:
prompt_template = f"Please answer the following question:\n{question_2}"

raw_result_2 = my_llm.invoke(prompt_template)
print("answer for question_2 without a knowledge base:")
print(raw_result_2.content)


### 4.4 Summary of the results

From the two questions above we can see that with the knowledge base connected, the model prefers to answer VMAX domain questions from the local documents; when the question is unrelated to the knowledge base, the prompt constrains the model to answer "I don't know". Without the knowledge base the model can only rely on its parametric knowledge and easily produces generic or even inaccurate answers. This is exactly the core value of RAG: mitigating hallucination and supplying private knowledge.


## 5. Adding conversation memory

The basic retrieval chain only handles the current question each time. If the user asks something that depends on the previous turn, such as "now put it into a table" or "simplify the result above", you need conversation history.

The following demonstrates the simplest multi-turn question answering with `ConversationalRetrievalChain`. Note: this uses the teaching API from `langchain_classic`, which emits a deprecation warning in the current version but still runs fine for this course.


In [ ]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)


### 5.1 Conversational retrieval chain (ConversationalRetrievalChain)

A conversational retrieval chain first uses the history to rewrite or understand the current question, then retrieves and answers. It suits demos where "the second question depends on the first answer".


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain

template = """You are the VMAX operations assistant. Answer using the conversation history and the context below.

Conversation history:
{chat_history}

Relevant context:
{context}

Question: {question}

Answer requirements:
1. Answer only from the context; never invent information that is not in it;
2. If the context does not contain the answer, reply "I don't know";
3. End with "Thanks for your question!".

Answer:"""

QA_PROMPT = PromptTemplate(
    input_variables=["chat_history", "context", "question"],
    template=template,
)

conversation_qa_chain = ConversationalRetrievalChain.from_llm(
    llm=my_llm,
    retriever=base_retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    chain_type="stuff",
)


In [ ]:
questions = [
    "\u4ec0\u4e48\u662fZXVMAX-S\u76845GC\u4e1a\u52a1\uff1f\u5305\u542b\u54ea\u4e9b\u529f\u80fd\uff1f/no_think",
    "\u5305\u542b\u54ea\u4e9b\u529f\u80fd\uff1f\u5bf9\u6bcf\u4e2a\u529f\u80fd\u505a\u4e0b\u9610\u8ff0/no_think",
    "\u5c06\u4ee5\u4e0a\u7ed3\u679c\u6574\u7406\u6210markdown\u8868\u683c\u5f62\u5f0f/no_think",
]

for question in questions:
    result = conversation_qa_chain.invoke({"question": question})
    print(f"question: {question}")
    print(f"answer: {result['answer']}")
    print("\n" + "=" * 50 + "\n")


### 5.2 Reading the conversation chain results

The first question retrieves from the `ZXVMAXS` simple vector store in Milvus and produces an answer about 5GC services; the next two questions do not repeat the full background and only say "the result above", so the chain keeps generating the table using the history of questions and answers stored in `memory`. This shows that memory solves "continuity of multi-turn context" while the vector database solves "retrieval of external knowledge" - two different responsibilities.


# Advanced RAG: query understanding and context optimisation

Basic RAG follows "retrieve with the raw question, feed the retrieved results straight into the LLM". This chapter optimises the two stages before and after retrieval:

1. **Query understanding**: normalise the user question, identify the business type, and decide between dense and hybrid retrieval;
2. **Context optimisation**: tidy the retrieved results, remove duplication, fill in adjacent semantics, and control the context length sent to the LLM.

The overall pipeline:

`user question -> query normalisation -> condition recognition and retrieval routing -> document recall -> deduplication and adjacency expansion -> context budget control -> LLM`

> This chapter first prepares two reusable optimisation modules; the later "hybrid retrieval" and "Agentic RAG" chapters plug them into the full pipeline.


## 1. Query understanding

### 1.1 Goal and processing strategy

Query understanding happens before document recall; its goal is to turn the user's natural phrasing into stable, executable retrieval conditions. This section does the following in order:

- **Query normalisation**: unify product aliases and abbreviations, for example mapping `vmax` to `ZXVMAX-S`;
- **Metadata filter condition recognition**: extract `business_type` from the question in preparation for later metadata filtering;
- **Retrieval routing**: choose Hybrid when the question contains alarm codes, interface names, or exact terms, and Dense for ordinary semantic questions.

**Related foundations:** the metadata filtering in the foundations module C04 (retrieval optimisation), lesson 02 (query construction), and the query rewriting and routing in the same module's lesson 04 (query reformulation and dispatch).

This section outputs a single query strategy dictionary: `original_query`, `normalized_query`, `business_type` and `route`.


### 1.2 Implementing query understanding

Below, query normalisation, business type recognition, and retrieval routing are wrapped into a single `prepare_retrieval_query()`.


In [ ]:
import re

QUERY_ALIASES = {
    "vmax": "ZXVMAX-S",
    "zxvmax": "ZXVMAX-S",
    "5g\u6838\u5fc3\u7f51": "5GC",
}
BUSINESS_TYPES = ["\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1", "\u8bed\u97f3\u4e1a\u52a1", "\u6570\u636e\u4e1a\u52a1", "\u7aef\u5230\u7aef\u4e1a\u52a1", "5GC\u4e1a\u52a1"]
BUSINESS_KEYWORDS = {
    "\u4e0a\u7f51\u65e5\u5fd7": "\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1",
    "\u8bed\u97f3": "\u8bed\u97f3\u4e1a\u52a1",
    "\u7aef\u5230\u7aef": "\u7aef\u5230\u7aef\u4e1a\u52a1",
    "5GC": "5GC\u4e1a\u52a1",
}

def normalize_query(question: str) -> str:
    normalized = re.sub(r"\s+", " ", question).strip()
    for alias in sorted(QUERY_ALIASES, key=len, reverse=True):
        canonical = QUERY_ALIASES[alias]
        if re.fullmatch(r"[A-Za-z0-9-]+", alias):
            pattern = rf"(?<![A-Za-z0-9-]){re.escape(alias)}(?![A-Za-z0-9-])"
        else:
            pattern = re.escape(alias)
        normalized = re.sub(pattern, canonical, normalized, flags=re.IGNORECASE)
    return normalized

def extract_query_business_type(question: str):
    compact = question.replace(" " , "")
    for business_type in BUSINESS_TYPES:
        if business_type.replace(" " , "") in compact:
            return business_type
    for keyword, business_type in BUSINESS_KEYWORDS.items():
        if keyword.upper() in question.upper():
            return business_type
    if "5GC" in question.upper() or "5G\u6838\u5fc3\u7f51" in question:
        return "5GC\u4e1a\u52a1"
    return None

def choose_retrieval_route(question: str) -> str:
    keyword_pattern = "(?:\\d{6,}|[A-Z]{2,}[A-Z0-9_-]*|\u63a5\u53e3|\u5b57\u6bb5|\u544a\u8b66|\u6a21\u5757|\u65e5\u5fd7)"
    return "hybrid" if re.search(keyword_pattern, question) else "dense"

def prepare_retrieval_query(question: str) -> dict:
    normalized = normalize_query(question)
    return {
        "original_query": question,
        "normalized_query": normalized,
        "business_type": extract_query_business_type(normalized),
        "route": choose_retrieval_route(normalized),
    }

query_strategy_examples = [
    prepare_retrieval_query("vmax \u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f"),
    prepare_retrieval_query("114020039 \u544a\u8b66\u600e\u4e48\u5904\u7406\uff1f"),
    prepare_retrieval_query("\u4e3a\u4ec0\u4e48\u9700\u8981\u505a\u4e1a\u52a1\u8d28\u91cf\u5b9a\u754c\uff1f"),
]
query_strategy_examples

### 1.3 Reading the results

The three examples show alias normalisation, exact-identifier routing, and plain semantic routing respectively. Only the retrieval strategy is produced here, no retrieval is executed yet; `business_type` can feed the metadata filter, and `route` decides in later chapters whether the Dense or the Hybrid retriever is called.


## 2. Context optimisation

### 2.1 Goal and processing strategy

Context optimisation happens after document recall and before the LLM call. This section processes the candidate documents in this order:

1. **Deduplication**: remove duplicated or heavily overlapping chunks;
2. **Adjacency expansion**: use `source_name + chunk_index` to add the content around a hit chunk, reducing semantic breaks;
3. **Budget control**: select documents within the `max_chars` limit so the context cannot grow without bound.

**Related foundations:** the context expansion in the foundations module C03 (index construction), lesson 05 (index optimisation), and the context compression idea in module C04 (retrieval optimisation), lesson 05 (advanced retrieval techniques).

> Adjacency expansion relies on the `source_name` and `chunk_index` metadata written during the optimisation preprocessing stage.


### 2.2 Implementing context optimisation

Below we implement document deduplication, adjacent chunk expansion, and context budget control separately.


In [ ]:
def document_identity(doc):
    return (
        doc.metadata.get("source_name") or doc.metadata.get("source"),
        doc.metadata.get("page_number", doc.metadata.get("page")),
        doc.metadata.get("section_title"),
        re.sub(r"\s+", "", doc.page_content)[:160],
    )

def deduplicate_documents(docs):
    seen = set()
    result = []
    for doc in docs:
        identity = document_identity(doc)
        if identity not in seen:
            seen.add(identity)
            result.append(doc)
    return result

def build_chunk_lookup(all_docs):
    return {
        (doc.metadata.get("source_name"), doc.metadata.get("chunk_index")): doc
        for doc in all_docs
        if doc.metadata.get("source_name") is not None
        and doc.metadata.get("chunk_index") is not None
    }

def expand_adjacent_chunks(docs, all_docs, window=1):
    lookup = build_chunk_lookup(all_docs)
    expanded = []
    for doc in docs:
        source_name = doc.metadata.get("source_name")
        chunk_index = doc.metadata.get("chunk_index")
        if source_name is None or chunk_index is None:
            expanded.append(doc)
            continue
        for neighbor_index in range(max(0, chunk_index - window), chunk_index + window + 1):
            neighbor = lookup.get((source_name, neighbor_index))
            if neighbor is not None:
                expanded.append(neighbor)
    return deduplicate_documents(expanded)

def select_context_with_budget(docs, max_chars=6000):
    selected, used = [], 0
    for doc in deduplicate_documents(docs):
        remaining = max_chars - used
        if remaining <= 0:
            break
        if len(doc.page_content) <= remaining:
            selected.append(doc)
            used += len(doc.page_content)
    return selected

print("context optimisation functions ready: deduplicate -> neighbour expansion -> budget control")

### 2.3 Demonstrating the effect of context optimisation

The following uses one real question to show `raw recall -> deduplication -> adjacency expansion -> budget control` end to end. To mimic the common case of multi-channel recall such as Dense plus BM25, the code deliberately copies the first result to create a duplicate chunk.

> The demo depends on the `base_retriever` created earlier and the `split_docs` produced by the optimisation preprocessing.


In [ ]:
context_demo_query = "ZXVMAX-S\u7684\u6570\u636e\u53ef\u9760\u6027\u5982\u4f55\u4fdd\u969c\uff1f"

# take 3 real recalled results, then copy the first one to simulate duplicate chunks from multi-path recall.
retrieved_docs = base_retriever.invoke(context_demo_query)[:3]
raw_demo_docs = retrieved_docs + retrieved_docs[:1]
deduplicated_demo_docs = deduplicate_documents(raw_demo_docs)

# prefer recalled chunks that carry complete metadata; fall back to split_docs when an older vector store lacks the fields.
expand_seed_docs = [
    doc for doc in deduplicated_demo_docs
    if doc.metadata.get("source_name") is not None
    and doc.metadata.get("chunk_index") is not None
][:2]
if not expand_seed_docs:
    expand_seed_docs = [
        doc for doc in split_docs
        if "\u6570\u636e\u53ef\u9760\u6027" in doc.metadata.get("section_title", "")
    ][:1]

expanded_demo_docs = expand_adjacent_chunks(expand_seed_docs, split_docs, window=1)
final_demo_docs = select_context_with_budget(expanded_demo_docs, max_chars=1800)

def summarize_context_stage(stage_name, docs):
    total_chars = sum(len(doc.page_content) for doc in docs)
    print(f"\n===== {stage_name}: {len(docs)} chunks / {total_chars} characters =====")
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "\u672a\u77e5\u6765\u6e90")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "\u672a\u8bc6\u522b\u7ae0\u8282")
        chunk_index = doc.metadata.get("chunk_index", "-")
        preview = re.sub(r"\s+", " ", doc.page_content)[:90]
        print(f"{i}. chunk={chunk_index} | page={page} | section={section}")
        print(f"   source={source}")
        print(f"   {preview}...")

print("demo question:", context_demo_query)
summarize_context_stage("before optimisation (with simulated duplicate chunks)", raw_demo_docs)
summarize_context_stage("step 1: after deduplication", deduplicated_demo_docs)
summarize_context_stage("step 2: after neighbour expansion", expanded_demo_docs)
summarize_context_stage("step 3: after the 1800-character budget", final_demo_docs)

print("\n===== before and after =====")
print(f"duplicate chunks removed: {len(raw_demo_docs)} -> {len(deduplicated_demo_docs)}")
print(f"neighbour expansion: {len(expand_seed_docs)} -> {len(expanded_demo_docs)}")
print(f"budget control: {sum(len(d.page_content) for d in expanded_demo_docs)} -> "
      f"{sum(len(d.page_content) for d in final_demo_docs)} characters")

### 2.4 How to read the demo results

While running, watch for:

- **the document count drops after deduplication**: duplicate chunks no longer occupy LLM context;
- **consecutive `chunk_index` values appear after adjacency expansion**: the semantics around the hit chunk are filled in;
- **nothing exceeds 1800 characters after budget control**: the final context length and call cost stay under control.

The temporary increase in document count caused by adjacency expansion is expected: expansion completes the semantics, and the final budget control limits the length.


## 3. Combining query understanding and context optimisation

Query understanding and context optimisation are not two separate RAG systems, but enhancement modules at the two ends of the full retrieval pipeline:

| Stage | Input | Core processing | Output |
| --- | --- | --- | --- |
| Query understanding | raw user question | normalisation, condition recognition, routing | query strategy |
| Retrieval execution | query strategy | Dense / Hybrid recall, metadata filter | candidate documents |
| Context optimisation | candidate documents | deduplication, adjacency expansion, budget control | final context |
| Answer generation | final context + original question | prompt + LLM | knowledge-grounded answer |

How the later chapters connect:

- **Hybrid retrieval**: consumes the `route` produced by query understanding and applies context optimisation to the recalled results;
- **Agentic RAG**: the Agent decides whether to call the retrieval tool, and the tool can still reuse query understanding and context optimisation internally;
- **Basic RAG**: kept as the baseline so the effect before and after these optimisations can be compared.


# Advanced RAG: hybrid retrieval

Basic RAG only uses `dense` semantic vector retrieval. It works well for natural-language questions but is not necessarily stable for literal keywords such as product names, alarm codes, interface names, and abbreviations. This chapter uses the `ZXVMAXS_HYBRID` collection built in the previous chapter to demonstrate **dense semantic retrieval + sparse BM25 keyword retrieval**, and then adds rerank precision ranking on top.

The chapter still runs teaching experiments over the VMAX documents, focusing on three things:

1. how hybrid retrieval fuses semantic and keyword recall;
2. the difference between the weighted and RRF fusion strategies;
3. how rerank re-selects the most suitable context to feed the LLM from the recalled candidates.


## 1. How hybrid retrieval works

Hybrid retrieval usually has two recall paths:

- **Dense vector retrieval**: encodes the query and document chunks into dense embeddings and recalls by semantic similarity; suits natural-language phrasing, synonymous rewrites, and vague questions;
- **Sparse / BM25 retrieval**: recalls based on keywords, term frequency and inverse document frequency; suits exact matches on product names, module names, alarm codes, interface names, abbreviations, and numeric identifiers.

Milvus 2.6 can generate sparse vectors on the server side through `BM25BuiltInFunction`. That means the client only writes `text` and the `dense` embedding, and Milvus automatically derives the `sparse` field from `text` and uses it in hybrid retrieval.

Note: `sparse` is the sparse vector field output by the BM25 function; it can only be used for retrieval scoring and cannot be returned as raw data. Therefore, before using the LangChain Milvus retriever, exclude `dense/sparse/vector` from `vectorstore.fields`.


## 2. A teaching-grade hybrid retrieval implementation

Before calling Milvus's built-in hybrid retrieval, we take "hybrid" apart with some teaching code so you can see it clearly:

1. **Dense path**: semantic vector retrieval from the simple store `ZXVMAXS`;
2. **BM25 path**: build local BM25 keyword retrieval from the same batch of document texts;
3. **Score normalisation**: squash both score families into 0-1;
4. **Weighted fusion**: `hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score`;
5. **Ranked output**: observe how the results change with different weights.

This code is not the recommended production approach; its purpose is to let learners see the underlying logic of hybrid retrieval. Once you understand it, using the server-side BM25 and ranker built into Milvus feels much more natural.


### 2.1 Preparing the Dense and BM25 recall paths

Here BM25 is built from the text field of the `ZXVMAXS` simple retrieval collection. For classroom speed we read at most 500 texts; the current sample data has about 322 rows, so it is read in full.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
import re

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)
connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

# Dense semantic search path: use the simple retrieval collection ZXVMAXS.
vectordb_dense = Milvus(
    embedding_function=my_emb,
    collection_name="ZXVMAXS",
    connection_args=connection_args,
    vector_field="dense",
    text_field="text",
)
vectordb_dense.fields = [field for field in vectordb_dense.fields if field not in {"dense", "sparse", "vector"}]

# BM25 keyword path: read text/source/page from the same collection and build a local BM25 index.
raw_rows = _milvus_client.query(
    collection_name="ZXVMAXS",
    filter="pk >= 0",
    output_fields=["text", "source", "file_path", "page"],
    limit=500,
)

bm25_docs = [
    Document(
        page_content=row["text"],
        metadata={
            "source": row.get("source") or row.get("file_path") or "\u672a\u77e5\u6765\u6e90",
            "page": row.get("page", "-"),
        },
    )
    for row in raw_rows
]

def tokenize_zh_en(text: str):
    """tokenizer: split words on English and digits, and use character bigrams for Chinese - enough to demo BM25."""
    text = text.lower()
    tokens = re.findall(r"[a-z0-9][a-z0-9_-]+", text)
    chinese = "".join(re.findall(r"[\u4e00-\u9fff]+", text))
    tokens.extend(chinese[i:i + 2] for i in range(max(len(chinese) - 1, 0)))
    return [token for token in tokens if len(token) >= 2]

bm25_corpus = [tokenize_zh_en(doc.page_content) for doc in bm25_docs]
bm25_index = BM25Okapi(bm25_corpus)

print(f"Dense collection: ZXVMAXS")
print(f"BM25 document count: {len(bm25_docs)}")


### 2.2 Fixed-weight fusion

Fixed weights suit the lesson "how are hybrid retrieval scores combined". Below we set `dense_weight=0.6` and `bm25_weight=0.4` by hand, representing the importance of semantic and keyword retrieval respectively.


In [ ]:
def normalize_scores(score_map):
    if not score_map:
        return {}
    values = list(score_map.values())
    min_score, max_score = min(values), max(values)
    if max_score == min_score:
        return {key: 1.0 for key in score_map}
    return {key: (value - min_score) / (max_score - min_score) for key, value in score_map.items()}

def doc_key(doc):
    source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
    page = str(doc.metadata.get("page", "-")).strip()
    return source, page, doc.page_content[:80]

def manual_hybrid_search(query, dense_weight=0.6, bm25_weight=0.4, dense_k=8, bm25_k=8, final_k=5):
    # Dense path: scored retrieval. The direction of the Milvus score depends on the distance metric, so the lesson min-max normalises it and only reads it as a relative ranking.
    dense_pairs = vectordb_dense.similarity_search_with_score(query, k=dense_k)
    dense_docs = [doc for doc, score in dense_pairs]
    dense_raw = {doc_key(doc): float(score) for doc, score in dense_pairs}
    doc_by_key = {doc_key(doc): doc for doc in dense_docs}

    # BM25 path: local keyword retrieval, where a higher score means more relevant.
    tokenized_query = tokenize_zh_en(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)
    top_bm25_indices = sorted(range(len(bm25_scores)), key=lambda idx: bm25_scores[idx], reverse=True)[:bm25_k]
    bm25_raw = {}
    for idx in top_bm25_indices:
        doc = bm25_docs[idx]
        key = doc_key(doc)
        bm25_raw[key] = float(bm25_scores[idx])
        doc_by_key.setdefault(key, doc)

    dense_norm = normalize_scores(dense_raw)
    bm25_norm = normalize_scores(bm25_raw)

    fused = []
    for key, doc in doc_by_key.items():
        dense_score = dense_norm.get(key, 0.0)
        bm25_score = bm25_norm.get(key, 0.0)
        hybrid_score = dense_weight * dense_score + bm25_weight * bm25_score
        fused.append((hybrid_score, dense_score, bm25_score, doc))

    fused.sort(key=lambda item: item[0], reverse=True)
    return fused[:final_k]

def print_manual_hybrid_results(title, results):
    print(f"\n===== {title} =====")
    for i, (hybrid_score, dense_score, bm25_score, doc) in enumerate(results, 1):
        source = doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source"
        page = doc.metadata.get("page", "-")
        preview = doc.page_content[:160].replace("\n", " ")
        print(f"{i}. hybrid={hybrid_score:.3f} | dense={dense_score:.3f} | bm25={bm25_score:.3f} | source={source} | page={page}")
        print(f"   {preview}")

manual_query = "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f"
fixed_results = manual_hybrid_search(manual_query, dense_weight=0.6, bm25_weight=0.4)
print_manual_hybrid_results("teaching build: fixed-weight hybrid retrieval", fixed_results)


### 2.3 Dynamic-weight fusion

Dynamic weights build on fixed weights and automatically adjust the dense/BM25 ratio from the characteristics of the query:

- questions with obvious keywords such as product names, alarm codes, interface names, and field names raise the BM25 weight;
- semantic questions such as root-cause analysis, optimisation advice, and impact explanations raise the dense weight;
- other questions stay balanced.


In [ ]:
def choose_manual_hybrid_weights(question: str):
    keyword_patterns = [
        r"\d{4,}",
        r"ZXVMAX-S",
        "\u63a5\u53e3|\u5b57\u6bb5|\u544a\u8b66|\u65e5\u5fd7|NAT|IMSI|MSISDN|URL",
        "\u6709\u54ea\u4e9b|\u5305\u62ec\u54ea\u4e9b|\u5217\u51fa|\u8868\u683c",
    ]
    semantic_patterns = [
        "\u4e3a\u4ec0\u4e48|\u539f\u56e0|\u5982\u4f55\u4f18\u5316|\u600e\u4e48\u89e3\u51b3|\u5f71\u54cd|\u4f18\u7f3a\u70b9|\u89e3\u91ca",
    ]
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return 0.4, 0.6, "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return 0.7, 0.3, "semantic_first"
    return 0.5, 0.5, "balanced"

for test_query in [
    "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f",
    "\u4e3a\u4ec0\u4e48\u9700\u8981\u505a5GC\u4e1a\u52a1\u8d28\u91cf\u5b9a\u754c\uff1f",
    "114020039\u544a\u8b66\u5982\u4f55\u5904\u7406\uff1f",
]:
    dense_weight, bm25_weight, query_type = choose_manual_hybrid_weights(test_query)
    results = manual_hybrid_search(
        test_query,
        dense_weight=dense_weight,
        bm25_weight=bm25_weight,
        final_k=3,
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, dense_weight={dense_weight}, bm25_weight={bm25_weight}")
    print_manual_hybrid_results("teaching build: dynamic-weight hybrid retrieval", results)


## 3. Milvus built-in hybrid retrieval

The hand-written version above exists to explain the principle, but pulling the BM25 index down to maintain it locally is not recommended in engineering. Milvus 2.6 can generate the sparse field on the server through `BM25BuiltInFunction` and perform weighted or RRF fusion directly at search time.

From here we start using the `ZXVMAXS_HYBRID` built in the previous chapter, which is the recommended way to call it in practice.


### 3.1 Loading the hybrid retrieval vector store

Here we load the `ZXVMAXS_HYBRID` collection created in the previous chapter. It contains two vector fields:

- `dense`: the semantic embedding field;
- `sparse`: the keyword retrieval field generated by Milvus's built-in BM25 function.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings

my_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

_milvus_client = MilvusClient(**connection_args)

vectordb_hybrid = Milvus(
    embedding_function=my_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)

# retrieval returns only text and metadata fields, never the raw dense/sparse vectors.
vectordb_hybrid.fields = [
    field for field in vectordb_hybrid.fields
    if field not in {"dense", "sparse", "vector"}
]

print("vector fields:", vectordb_hybrid.vector_fields)
print("returned fields:", vectordb_hybrid.fields)


### 3.2 Weighted fusion

The weighted ranker fuses the dense and sparse scores by weight. Below we set `weights=[0.6, 0.4]`:

- `0.6` goes to dense semantic retrieval;
- `0.4` goes to sparse BM25 keyword retrieval.

Raise the dense weight when the question is more natural-language; raise the sparse weight when it contains keywords such as alarm codes, product names, or interface names.


In [ ]:
def print_docs(title, docs, preview_chars=220):
    print(f"\n===== {title} =====")
    displayed = set()
    row = 1
    for doc in docs:
        source = str(doc.metadata.get("source") or doc.metadata.get("file_path") or "unknown source").strip()
        page = str(doc.metadata.get("page", "-")).strip()
        key = (source, page, doc.page_content[:80])
        if key in displayed:
            continue
        displayed.add(key)
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        score = doc.metadata.get("rerank_score")
        score_text = f" | rerank_score={score:.2f}" if isinstance(score, (int, float)) else ""
        print(f"{row}. source={source} | page={page}{score_text}")
        print(f"   {preview}")
        row += 1

query = "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f"

weighted_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("weighted hybrid retrieval results", weighted_docs)


### 3.3 RRF fusion

RRF (Reciprocal Rank Fusion) needs no manual dense/sparse weights; it fuses based on the rank position within each recall list. It suits teaching and early experiments: when you are unsure how to set the weights, start with RRF as a robust baseline.


In [ ]:
rrf_docs = vectordb_hybrid.similarity_search(
    query,
    k=5,
    ranker_type="rrf",
    ranker_params={"k": 100},
)

print_docs("RRF hybrid retrieval results", rrf_docs)


### 3.4 The dynamic weight strategy

In real business, different questions suit different weights:

- questions containing keywords such as alarm codes, interface names, product names, field names, and identifiers suit a higher BM25 sparse weight;
- questions phrased more naturally that need semantic understanding suit a higher dense weight;
- when unsure, use balanced weights or go straight to RRF.

The following demonstrates dynamic weighting with a simple rule. The rule is not a final production design, but it helps students understand why the weights matter.


In [ ]:
import re

def choose_hybrid_weights(question: str):
    keyword_patterns = [
        "\\d{4,}",          # long numeric strings: alarm codes and identifiers
        "ZXVMAX-S",          # product name
        "\u63a5\u53e3|\u5b57\u6bb5|\u544a\u8b66|\u65e5\u5fd7|NAT|IMSI|MSISDN|URL",
        "\u6709\u54ea\u4e9b|\u5305\u62ec\u54ea\u4e9b|\u5217\u51fa|\u8868\u683c",
    ]
    semantic_patterns = [
        "\u4e3a\u4ec0\u4e48|\u539f\u56e0|\u5982\u4f55\u4f18\u5316|\u600e\u4e48\u89e3\u51b3|\u5f71\u54cd|\u4f18\u7f3a\u70b9|\u89e3\u91ca",
    ]

    if any(re.search(pattern, question, re.IGNORECASE) for pattern in keyword_patterns):
        return [0.4, 0.6], "keyword_first"
    if any(re.search(pattern, question, re.IGNORECASE) for pattern in semantic_patterns):
        return [0.7, 0.3], "semantic_first"
    return [0.5, 0.5], "balanced"

for test_query in [
    "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f",
    "\u4e3a\u4ec0\u4e48\u9700\u8981\u505a5GC\u4e1a\u52a1\u8d28\u91cf\u5b9a\u754c\uff1f",
    "114020039\u544a\u8b66\u5982\u4f55\u5904\u7406\uff1f",
]:
    weights, query_type = choose_hybrid_weights(test_query)
    docs = vectordb_hybrid.similarity_search(
        test_query,
        k=3,
        ranker_type="weighted",
        ranker_params={"weights": weights},
    )
    print(f"\nQuery: {test_query}")
    print(f"query_type={query_type}, weights={weights}")
    print_docs("dynamic-weight retrieval results", docs, preview_chars=120)


## 4. Rerank precision ranking

Hybrid retrieval solves "recall as many plausibly relevant candidates as possible", but the candidate list still contains redundant or weakly related content. Rerank scores each query-document pair and re-sorts, typically used as:

1. recall more candidates with hybrid search first, for example top 10 or top 20;
2. rerank them and keep the top 3 to top 5;
3. hand the high-quality reranked context to the LLM.

This section compares three rerank approaches and completes the real test with the Qwen3 Reranker deployed on Xinference:

- **Cohere Rerank**: a cloud API, mature quality, requires `COHERE_API_KEY`;
- **Jina Reranker**: a cloud HTTP API with multilingual support, requires `JINA_API_KEY`;
- **Xinference Qwen3 Reranker**: calls `my_qwen3_reranker_0.6b` through Xinference's OpenAI-compatible `{XINFERENCE_OPENAI_BASE_URL}/rerank` endpoint, returning ranking indexes and relevance scores directly, which suits intranet, private, and teaching environments.


### 4.1 Cohere Rerank (example only)

If `COHERE_API_KEY` is configured in your environment you can use `langchain_cohere.CohereRerank`. Without an API key, skip this cell.


In [ ]:
# import os

# if os.getenv("COHERE_API_KEY"):
#     from langchain_cohere import CohereRerank

#     cohere_compressor = CohereRerank(
#         model="rerank-multilingual-v3.0",
#         top_n=3,
#     )
#     cohere_reranked_docs = cohere_compressor.compress_documents(weighted_docs, query=query)
#     print_docs("Cohere Rerank results", cohere_reranked_docs)
# else:
#     print("COHERE_API_KEY is not configured, skipping the Cohere Rerank example.")


### 4.2 Jina Reranker (example only)

Jina Reranker can be called straight through an HTTP API. Here is a generic template: if `JINA_API_KEY` is configured it actually calls the service, otherwise it only prints a skip message.


In [ ]:
# import os
# import httpx
# from langchain_core.documents import Document

# def jina_rerank(query, documents, top_n=3, model="jina-reranker-v2-base-multilingual"):
#     api_key = os.getenv("JINA_API_KEY")
#     if not api_key:
#         print("JINA_API_KEY is not configured, skipping the Jina Rerank example.")
#         return []

#     payload = {
#         "model": model,
#         "query": query,
#         "top_n": top_n,
#         "documents": [doc.page_content for doc in documents],
#     }
#     headers = {
#         "Authorization": f"Bearer {api_key}",
#         "Content-Type": "application/json",
#     }
#     response = httpx.post(
#         "https://api.jina.ai/v1/rerank",
#         headers=headers,
#         json=payload,
#         timeout=60,
#         trust_env=False,
#     )
#     response.raise_for_status()
#     results = response.json().get("results", [])

#     reranked = []
#     for item in results:
#         idx = item["index"]
#         score = item.get("relevance_score", 0.0)
#         doc = documents[idx]
#         reranked.append(Document(
#             page_content=doc.page_content,
#             metadata={**doc.metadata, "rerank_score": float(score), "rerank_model": model},
#         ))
#     return reranked

# jina_reranked_docs = jina_rerank(query, weighted_docs, top_n=3)
# if jina_reranked_docs:
#     print_docs("Jina Rerank results", jina_reranked_docs)


### 4.3 Xinference Qwen3 Reranker (recommended)

Xinference publishes reranker models as an OpenAI-compatible relevance endpoint at `{XINFERENCE_OPENAI_BASE_URL}/rerank`,
whose request body is essentially the same as Cohere's `/v2/rerank`: `model` / `query` / `documents` / `top_n`, returning `results[].index` and `results[].relevance_score`.

Compared with using a generation endpoint to "have the model mentally estimate a score", a dedicated rerank endpoint scores the whole candidate batch in one request: lower latency, comparable scores, and the returned indexes never drift out of sync with the original documents.

Two things to watch in the implementation:

1. what you send to the service is the list of `Document.page_content`, and `index` in the response maps results back to the original LangChain `Document` objects;
2. keep the original document metadata while adding `rerank_score` and `rerank_model`, which makes later debugging, display, and evaluation easier.

> **How to read the output**: the relevance scores from Qwen3 Reranker clearly saturate towards `1.0`, so when teaching focus on **whether the ordering is sensible** and do not treat the absolute score as a threshold.


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain_core.documents import Document

_ = load_dotenv(find_dotenv())

# reuse the reranker configuration from the shared configuration cell, falling back to .env
RERANKER_BASE_URL = globals().get("RERANKER_BASE_URL") or os.environ["XINFERENCE_OPENAI_BASE_URL"]
RERANKER_MODEL = globals().get("RERANKER_MODEL") or os.environ["XINFERENCE_RERANKER_MODEL_ID"]


def xinference_qwen3_rerank(query, documents, top_n=3):
    """Call Xinference /v1/rerank to rerank the candidate documents and return them sorted by descending relevance."""
    if not documents or top_n <= 0:
        return []

    response = httpx.post(
        RERANKER_BASE_URL.rstrip("/") + "/rerank",
        headers={"Authorization": "Bearer " + os.environ["XINFERENCE_API_KEY"]},
        json={
            "model": RERANKER_MODEL,
            "query": query,
            "documents": [doc.page_content for doc in documents],
            "top_n": min(top_n, len(documents)),
        },
        timeout=60,
        trust_env=False,
    )
    response.raise_for_status()

    reranked_docs = []
    for item in response.json().get("results", []):
        doc = documents[item["index"]]
        reranked_docs.append(Document(
            page_content=doc.page_content,
            metadata={
                **doc.metadata,
                "rerank_score": float(item.get("relevance_score", 0.0)),
                "rerank_model": RERANKER_MODEL,
            },
        ))

    reranked_docs.sort(key=lambda item: item.metadata.get("rerank_score", 0.0), reverse=True)
    return reranked_docs

## 5. Real test: hybrid retrieval plus Xinference rerank

Below the weighted hybrid retrieval first recalls 8 candidates, then the Qwen3 Reranker on Xinference re-ranks them and keeps 5. When teaching, compare the source pages and content snippets before and after rerank to see whether precision ranking moved the context closest to the question to the front.


In [ ]:
rerank_query = "\u4ec0\u4e48\u662fZXVMAX-S\u76845GC\u4e1a\u52a1\uff1f\u5305\u542b\u54ea\u4e9b\u529f\u80fd\uff1f"

candidate_docs = vectordb_hybrid.similarity_search(
    rerank_query,
    k=8,
    ranker_type="weighted",
    ranker_params={"weights": [0.6, 0.4]},
)

print_docs("before rerank: hybrid retrieval candidates", candidate_docs, preview_chars=140)

reranked_docs = xinference_qwen3_rerank(rerank_query, candidate_docs, top_n=5)
print_docs("after rerank: Xinference Qwen3 reranked results", reranked_docs, preview_chars=180)


## 6. Hybrid retrieval summary

The recommended practice path in this chapter is:

1. **Basic version**: get a working RAG running with dense vector retrieval first;
2. **Advanced version**: introduce Milvus's BM25 sparse field and do hybrid retrieval with weighted or RRF fusion;
3. **Quality optimisation**: when the recall is plentiful but noisy, add rerank after hybrid retrieval so the most relevant context ranks first;
4. **Model choice**: Cohere/Jina suit cloud API scenarios, the Xinference Qwen3 Reranker suits intranet, private, and teaching environments.

One thing to keep in mind: Milvus's `sparse` field is a retrieval field, not a display field. Whether or not you add rerank, what is finally returned in the LangChain document objects should be ordinary fields such as `text/source/page`.


## 7. The complete hybrid retrieval chain (with memory)

The hybrid retrieval and rerank above solve "recall and ordering", but a complete RAG teaching case also needs them wired into a real question-answering chain.

This subsection connects the capabilities prepared earlier into one complete pipeline:

1. **Query understanding**: after multi-turn rewriting, normalise the question again, identify the business type, and choose the Dense / Hybrid route;
2. **Retrieval and rerank**: call `ZXVMAXS` or `ZXVMAXS_HYBRID` according to the route, applying the metadata filter and optional rerank;
3. **Context optimisation**: deduplicate the recalled results, expand to adjacent chunks, and control the context budget;
4. **Generation and memory**: generate the answer with a shared template and support multi-turn dialogue through `RunnableWithMessageHistory`.


### 7.1 Designing the complete pipeline

Compared with the single-turn retrieval chain earlier, this adds a "conversation state":

- in the first turn the user asks which features the ZXVMAX-S internet-surfing log supports; the system retrieves first and then generates the answer;
- in the second turn the user asks how long the data can be retained, without repeating the subject; the system brings the previous turn into the current answer so the pronoun is not misresolved;
- if you want to show the internals in class, print the history and the rewritten retrieval query directly.

This style is closer to a real continuous Q&A assistant. The actual execution path is:

`history rewriting -> query normalisation -> business type recognition -> Dense / Hybrid routing -> metadata filter -> rerank -> deduplication -> adjacency expansion -> budget control -> answer generation`


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# reuse the retrieval stores created in earlier chapters; rebuild them from the shared configuration if missing.
qa_connection_args = globals().get("connection_args", {"uri": os.environ["MILVUS_URI"]})
qa_hybrid_db = globals().get("vectordb_hybrid")
if qa_hybrid_db is None:
    from pymilvus import MilvusClient
    from langchain_milvus import Milvus, BM25BuiltInFunction
    from langchain_openai import OpenAIEmbeddings

    qa_emb = OpenAIEmbeddings(
        base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
        api_key=os.environ["XINFERENCE_API_KEY"],
        model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
    )
    _qa_milvus_client = MilvusClient(**qa_connection_args)

    qa_hybrid_db = Milvus(
        embedding_function=qa_emb,
        builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
        collection_name="ZXVMAXS_HYBRID",
        connection_args=qa_connection_args,
        vector_field=["dense", "sparse"],
        text_field="text",
        consistency_level="Bounded",
    )

# reuse the hybrid store that is already built, filtering the fields once more so sparse/dense are never returned as raw data.
qa_hybrid_db.fields = [
    field for field in qa_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

# the dense route reuses the simple vector store so query routing can genuinely pick a single retrieval path.
qa_dense_db = globals().get("vectordb_dense")
if qa_dense_db is None:
    qa_dense_db = globals().get("vectordb")
if qa_dense_db is None:
    from langchain_milvus import Milvus
    from langchain_openai import OpenAIEmbeddings
    qa_dense_emb = globals().get("my_emb")
    if qa_dense_emb is None:
        qa_dense_emb = OpenAIEmbeddings(
            base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
            api_key=os.environ["XINFERENCE_API_KEY"],
            model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
        )
    qa_dense_db = Milvus(
        embedding_function=qa_dense_emb,
        collection_name="ZXVMAXS",
        connection_args=qa_connection_args,
        vector_field="dense",
        text_field="text",
        consistency_level="Bounded",
    )
qa_dense_db.fields = [
    field for field in qa_dense_db.fields
    if field not in {"dense", "sparse", "vector"}
]

# reuse the LLM created in earlier chapters; rebuild it from the shared model configuration if this session has none.
qa_llm = globals().get("my_llm")
if qa_llm is None:
    import os
    import httpx
    from dotenv import load_dotenv, find_dotenv
    from langchain.chat_models import init_chat_model

    _ = load_dotenv(find_dotenv())
    if not os.getenv("vLLM_QWEN_API_KEY"):
        raise RuntimeError("vLLM_QWEN_API_KEY is not configured, so the retrieval QA chain example cannot run.")

    qa_llm = init_chat_model(
        model=os.environ["vLLM_QWEN_MODEL_ID"],
        model_provider="openai",
        base_url=os.environ["vLLM_QWEN_BASE_URL"],
        api_key=os.environ["vLLM_QWEN_API_KEY"],
        http_client=httpx.Client(trust_env=False),
        temperature=0.3,
        max_tokens=3000,
    )

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a retrieval query rewriter. Using the conversation history, rewrite the current question into a standalone, complete question suited to retrieval. Output only the rewritten question, no explanation.",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are the VMAX operations assistant. Answer only from the given context, never invent anything."
        "If the context does not contain the answer, reply \"I don't know\" explicitly."
        "Append the citation source and page number at the end of the answer.",
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        "original question: {question}\n\n"
        "rewritten retrieval question: {standalone_question}\n\n"
        "relevant context:\n{context}\n\n"
        "Give a concise and accurate answer in Chinese.",
    ),
])

question_rewriter = rewrite_prompt | qa_llm | StrOutputParser()


def _format_retrieval_docs(docs, preview_chars=260):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = str(doc.metadata.get("source_name") or doc.metadata.get("source") or doc.metadata.get("file_path") or "\u672a\u77e5\u6765\u6e90")
        page = str(doc.metadata.get("page_number", doc.metadata.get("page", "-")))
        section = str(doc.metadata.get("section_title", "\u672a\u8bc6\u522b\u7ae0\u8282"))
        preview = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page} | section={section}\n{preview}")
    return "\n\n".join(lines)


def _retrieve_by_route(question: str, route: str, k=12):
    if route == "hybrid":
        return qa_hybrid_db.similarity_search(
            question,
            k=k,
            ranker_type="weighted",
            ranker_params={"weights": [0.6, 0.4]},
        )
    return qa_dense_db.similarity_search(question, k=k)


def build_hybrid_qa_inputs(inputs: dict):
    question = inputs["question"]
    chat_history = inputs.get("chat_history", [])
    standalone_question = question
    if chat_history:
        standalone_question = question_rewriter.invoke({
            "question": question,
            "chat_history": chat_history,
        }).strip()

    # after rewriting, normalise the question and re-detect the conditions again, so a vague follow-up never keeps the old strategy.
    query_strategy = prepare_retrieval_query(standalone_question)
    standalone_question = query_strategy["normalized_query"]
    retrieval_route = query_strategy["route"]
    docs = _retrieve_by_route(standalone_question, retrieval_route, k=12)
    business_type = query_strategy["business_type"]
    if business_type:
        filtered_docs = [
            doc for doc in docs
            if doc.metadata.get("business_type") == business_type
        ]
        if filtered_docs:
            docs = filtered_docs

    if "xinference_qwen3_rerank" in globals():
        docs = xinference_qwen3_rerank(standalone_question, docs, top_n=5)

    docs = deduplicate_documents(docs)
    adjacent_source_docs = globals().get("split_docs", [])
    adjacent_expansion_applied = bool(adjacent_source_docs)
    if adjacent_expansion_applied:
        docs = expand_adjacent_chunks(docs[:3], adjacent_source_docs, window=1)
    docs = select_context_with_budget(docs, max_chars=6000)

    return {
        "question": question,
        "standalone_question": standalone_question,
        "retrieval_route": retrieval_route,
        "business_type": business_type,
        "adjacent_expansion_applied": adjacent_expansion_applied,
        "source_documents": docs,
        "chat_history": chat_history,
        "context": _format_retrieval_docs(docs),
    }

hybrid_qa_chain = (
    RunnableLambda(build_hybrid_qa_inputs)
    | answer_prompt
    | qa_llm
    | StrOutputParser()
)

conversation_store = {}


def get_hybrid_qa_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in conversation_store:
        conversation_store[session_id] = ChatMessageHistory()
    return conversation_store[session_id]

hybrid_qa_with_memory = RunnableWithMessageHistory(
    hybrid_qa_chain,
    get_hybrid_qa_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

print("hybrid retrieval QA chain is ready.")
print("- supports query normalisation and metadata filters")
print("- supports actual dense / hybrid routing")
print("- supports deduplication, neighbour expansion and budget control")
print("- supports optional Xinference rerank")
print("- supports conversation memory")


### 7.2 Single-turn question answering

First run one typical feature-type question to confirm the complete chain works out of the box.


In [ ]:
qa_session_id = "hybrid_qa_demo"
first_question = "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f"
first_answer = hybrid_qa_with_memory.invoke(
    {"question": first_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("user question:", first_question)
print("\nmodel answer:\n")
print(first_answer)

### 7.3 Multi-turn follow-up with memory

The second question deliberately leaves out a complete subject, so you can observe whether the system uses the previous turn to complete the question before retrieving.


In [ ]:
follow_up_question = "\u90a3\u5b83\u652f\u6301\u4fdd\u5b58\u591a\u4e45\uff1f"

# read the history of the current session first, and show the rewritten retrieval question while we are at it.
current_history = get_hybrid_qa_history(qa_session_id).messages
preview_inputs = {
    "question": follow_up_question,
    "chat_history": current_history,
}
preview_result = build_hybrid_qa_inputs(preview_inputs)

print("user follow-up:", follow_up_question)
print("rewritten retrieval question:", preview_result["standalone_question"])
print("retrieval route actually used:", preview_result["retrieval_route"])
print("business type detected:", preview_result["business_type"] or "unrestricted")
print("neighbour expansion:", "applied" if preview_result["adjacent_expansion_applied"] else "not applied (no split_docs in this session)")
print("\nretrieval context preview:\n")
print(preview_result["context"][:1400])

second_answer = hybrid_qa_with_memory.invoke(
    {"question": follow_up_question},
    config={"configurable": {"session_id": qa_session_id}},
)

print("\nmodel answer:\n")
print(second_answer)

### 7.4 Summary of the complete pipeline

This section turns "hybrid retrieval" into a genuinely conversational RAG chain:

- the rewriting layer turns follow-ups into stand-alone questions and re-runs query normalisation after rewriting;
- the routing layer identifies the business type and actually chooses Dense or Hybrid retrieval;
- the retrieval layer handles the metadata filter, candidate recall, and optional rerank;
- the context layer handles deduplication, adjacency expansion, and budget control;
- the memory layer keeps the conversation context;
- the generation layer organises the retrieved content into a stable answer.

At this point the hybrid retrieval chapter is more than a retrieval comparison: it covers the full "recall - precision ranking - generation - memory" loop.


# Advanced RAG: Agentic RAG

Both the basic RAG and the hybrid retrieval RAG earlier use a "developer-fixed flow": every user question enters one particular retrieval chain, and then the LLM answers from the retrieved results.

Agentic RAG instead wraps retrieval as tools and lets the Agent decide per question:

```text
user question -> Agent classifies the task -> selects a tool -> observes the tool result -> generates the final answer
```

This matches the Agent approach used in the LangChain agent notebook from the companion agent course; here the ordinary tools are upgraded into RAG tools:

- **Simple retrieval tool**: based on `ZXVMAXS`, dense semantic retrieval only, suits semantic questions;
- **Hybrid retrieval tool**: based on `ZXVMAXS_HYBRID`, fuses dense + BM25, suits questions heavy in product names, module names, alarm codes, and proper nouns;
- **Calculation tool**: demonstrates that an Agent does not always have to query the knowledge base and may complete the task with another tool.

The chapter focuses on three paths:

1. **Simple retrieval**: semantic descriptions, reason explanations, and similar questions call `simple_rag_search`;
2. **Hybrid retrieval**: questions with exact keywords such as product names, alarm codes, and interface names call `hybrid_rag_search`;
3. **No retrieval**: greetings, text rewriting, and general-capability questions that do not depend on enterprise knowledge are answered by the Agent directly, with no retrieval tool call.

For teaching, read it as: RAG is a capability, the Agent is the decision layer that schedules these capabilities.


## 1. Agentic RAG principles and tool design

The key to tool design is not "push all the logic into the Agent" but writing clear boundaries for each tool:

| Scenario | Agent behaviour | Suitable questions | Underlying capability |
|---|---|---|---|
| Simple retrieval | calls `simple_rag_search` | semantic descriptions, reason explanations, concept explanations | Milvus dense retrieval |
| Hybrid retrieval | calls `hybrid_rag_search` | product names, module names, alarm codes, interface names, field names | Milvus dense + BM25, optional rerank |
| No retrieval | answers directly without retrieval tools | greetings, text rewriting, general-capability questions | the LLM itself |
| Non-retrieval tool | calls `calculate` as needed | mathematical expressions | Python AST allowlist evaluation |

A tool's docstring goes into the model context and directly affects whether the Agent picks the right tool. So tool descriptions must be specific, restrained, and distinguishable.


## 2. Implementing the Agentic RAG tools

The code below reconnects the two already-built Milvus collections so that this section can run on its own:

- `ZXVMAXS`: the simple dense retrieval collection;
- `ZXVMAXS_HYBRID`: the hybrid retrieval collection with the two vector fields `dense` and `sparse`.

Note: Milvus vector fields exist for retrieval only and should not be returned as raw fields, so `dense/sparse/vector` is filtered out again here.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

from pymilvus import MilvusClient
from langchain_milvus import Milvus, BM25BuiltInFunction
from langchain_openai import OpenAIEmbeddings
from langchain_core.tools import tool

# this section rebuilds its own connection so it can be run starting from the Agentic RAG chapter.
agent_connection_args = {
    "uri": os.environ["MILVUS_URI"],
}

agent_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

_agent_milvus_client = MilvusClient(**agent_connection_args)

agent_simple_db = Milvus(
    embedding_function=agent_emb,
    collection_name="ZXVMAXS",
    connection_args=agent_connection_args,
    vector_field="dense",
    text_field="text",
    consistency_level="Bounded",
)
agent_simple_db.fields = [
    field for field in agent_simple_db.fields
    if field not in {"dense", "sparse", "vector"}
]

agent_hybrid_db = Milvus(
    embedding_function=agent_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=agent_connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
agent_hybrid_db.fields = [
    field for field in agent_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_tool_docs(docs, max_chars=320):
    """Compress the retrieval results into text that a tool can hand back to the agent."""
    lines = []
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "\u672a\u77e5\u6765\u6e90")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "\u672a\u8bc6\u522b\u7ae0\u8282")
        content = doc.page_content[:max_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page} | section={section}\n{content}")
    return "\n\n".join(lines)

def _filter_and_optimize_tool_docs(question, docs, use_rerank=False):
    strategy = prepare_retrieval_query(question)
    business_type = strategy["business_type"]
    if business_type:
        filtered = [doc for doc in docs if doc.metadata.get("business_type") == business_type]
        if filtered:
            docs = filtered
    if use_rerank and "xinference_qwen3_rerank" in globals():
        docs = xinference_qwen3_rerank(strategy["normalized_query"], docs, top_n=4)
    docs = deduplicate_documents(docs)
    adjacent_source_docs = globals().get("split_docs", [])
    if adjacent_source_docs:
        docs = expand_adjacent_chunks(docs[:3], adjacent_source_docs, window=1)
    return strategy, select_context_with_budget(docs, max_chars=5000)

@tool
def simple_rag_search(question: str) -> str:
    """Search the ZXVMAX-S knowledge base with simple dense vector retrieval. Best for semantic, descriptive and concept-explanation questions."""
    strategy = prepare_retrieval_query(question)
    docs = agent_simple_db.similarity_search(strategy["normalized_query"], k=8)
    strategy, docs = _filter_and_optimize_tool_docs(question, docs)
    if not docs:
        return "simple retrieval found no relevant material."
    return "retrieval path: single_dense\n" + _format_tool_docs(docs)

@tool
def hybrid_rag_search(question: str) -> str:
    """Search the ZXVMAX-S knowledge base with Milvus hybrid retrieval. Best for questions containing keywords such as product names, module names, alarm codes, interface names or field names."""
    strategy = prepare_retrieval_query(question)
    normalized_question = strategy["normalized_query"]
    docs = agent_hybrid_db.similarity_search(
        normalized_question,
        k=8,
        ranker_type="weighted",
        ranker_params={"weights": [0.5, 0.5]},
    )
    strategy, docs = _filter_and_optimize_tool_docs(question, docs, use_rerank=True)
    if not docs:
        return "hybrid retrieval found no relevant material."
    return "retrieval path: hybrid\n" + _format_tool_docs(docs)

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression safely. Supports the four arithmetic operations, exponentiation and parenthesised expressions. Do not use it for knowledge-base questions."""
    import ast
    import operator

    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
    }

    def safe_eval(node):
        if isinstance(node, ast.Expression):
            return safe_eval(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.left), safe_eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in operators:
            return operators[type(node.op)](safe_eval(node.operand))
        raise ValueError(f"unsupported expression: {ast.dump(node)}")

    try:
        tree = ast.parse(expression, mode="eval")
        return f"result: {expression} = {safe_eval(tree)}"
    except Exception as exc:
        return f"calculation error: {exc}"

agentic_rag_tools = [simple_rag_search, hybrid_rag_search, calculate]

print("Agentic RAG tools ready:")
for item in agentic_rag_tools:
    print("-", item.name, ":", item.description)

### 2.1 Testing the retrieval tools standalone

Before handing them to the Agent, call the tools directly. This has two benefits:

1. confirm that the tools connect to the external systems correctly;
2. let students see the raw content a tool returns before "the Agent calls a tool".

If the tool itself is broken, no matter how smart the Agent is it cannot produce a correct result.


In [ ]:
tool_test_questions = [
    (simple_rag_search, {"question": "\u4e3a\u4ec0\u4e48\u9700\u8981\u505a5GC\u4e1a\u52a1\u8d28\u91cf\u5b9a\u754c\uff1f"}),
    (hybrid_rag_search, {"question": "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f"}),
    (calculate, {"expression": "(15 * 37 + 228) / 3"}),
]

for tool_item, args in tool_test_questions:
    print("=" * 80)
    print("tool:", tool_item.name)
    print("input:", args)
    print(tool_item.invoke(args)[:1200])

## 3. Building the tool-calling Agent

The current environment uses LangChain 1.x, whose recommended way to build an Agent graph is `create_agent()`; `AgentExecutor` and `create_tool_calling_agent`, common in older tutorials, are no longer exported directly from `langchain.agents` in some newer versions.

The biggest difference from an ordinary RAG chain:

- ordinary RAG chain: the developer decides in advance which retriever is always used;
- Agentic RAG: the model first understands the question, then decides whether to call a tool, which tool, and how to answer from the tool result.

`debug` is off by default so the output stays readable as teaching material. To inspect model messages, tool calls, and tool results in class, set `DEBUG_AGENTIC_RAG` to `True` in the code.


In [ ]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

_ = load_dotenv(find_dotenv())
agent_api_key = os.getenv("vLLM_QWEN_API_KEY")


agentic_system_prompt = (
    "You are an enterprise knowledge-base Q&A assistant."
    "First decide whether the question depends on ZXVMAX-S enterprise knowledge, then decide whether to call a tool."
    "Case 1: enterprise knowledge questions that are semantic descriptions, cause explanations or concept explanations - call simple_rag_search and do not also call hybrid_rag_search."
    "Case 2: enterprise knowledge questions with exact keywords such as product names, module names, alarm codes, interface names, field names or identifiers - call hybrid_rag_search and do not also call simple_rag_search."
    "Case 3: greetings, text rewriting and general-capability questions that do not depend on enterprise knowledge - call no retrieval tool and answer directly."
    "Mathematical calculations call calculate; that is a non-retrieval path."
    "When answering with a retrieval tool you must rely on what the tool returned; if the evidence is thin, say explicitly that the knowledge base contains no sufficient evidence, and list the citation sources and page numbers."
    "When you do not use retrieval, never invent internal company information, and never fabricate citations."
)


agentic_llm = my_llm

DEBUG_AGENTIC_RAG = False

agentic_agent = create_agent(
        model=agentic_llm,
        tools=agentic_rag_tools,
        system_prompt=agentic_system_prompt,
        debug=DEBUG_AGENTIC_RAG,
    )

print("Agentic RAG agent created.")

## 4. Testing the Agentic RAG behaviour

Three core scenarios check whether the Agent picks the right path:

1. **Semantic explanation question**: should call `simple_rag_search` and use single dense retrieval;
2. **Exact keyword question**: should call `hybrid_rag_search` and use hybrid retrieval;
3. **Non-knowledge-base question**: should not call a retrieval tool and be answered by the Agent directly.

A calculation question is kept as an extra example: it may call `calculate`, but still counts as "no knowledge-base retrieval".

In a real project you should not judge the Agent's tool selection from a single demo; keep observing through LangSmith, logs, or an evaluation set.


In [ ]:
agentic_examples = [
    {"scene": "\u5355\u4e00\u68c0\u7d22", "question": "\u4e3a\u4ec0\u4e48\u9700\u8981\u505a\u4e1a\u52a1\u8d28\u91cf\u5b9a\u754c\uff1f", "expected_tools": ["simple_rag_search"]},
    {"scene": "\u6df7\u5408\u68c0\u7d22", "question": "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f", "expected_tools": ["hybrid_rag_search"]},
    {"scene": "\u4e0d\u8d70\u68c0\u7d22", "question": "\u4f60\u597d\uff0c\u8bf7\u7528\u4e00\u53e5\u8bdd\u4ecb\u7ecd\u4f60\u80fd\u505a\u4ec0\u4e48\u3002", "expected_tools": []},
    {"scene": "\u975e\u68c0\u7d22\u5de5\u5177\uff08\u6269\u5c55\uff09", "question": "\u5e2e\u6211\u8ba1\u7b97 (15 * 37 + 228) / 3\u3002", "expected_tools": ["calculate"]},
]

def print_agent_answer(result):
    """Handle the messages structure returned by create_agent in LangChain 1.x."""
    messages = result.get("messages", []) if isinstance(result, dict) else []
    if not messages:
        print(result)
        return
    final_message = messages[-1]
    content = getattr(final_message, "content", final_message)
    print(content)

def extract_called_tools(result):
    called = []
    for message in result.get("messages", []) if isinstance(result, dict) else []:
        for call in getattr(message, "tool_calls", []) or []:
            name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
            if name:
                called.append(name)
    return called

if "agentic_agent" not in globals():
    print("the Agentic agent was not created, usually because vLLM_QWEN_API_KEY is not configured.")
else:
    for example in agentic_examples:
        question = example["question"]
        print("=" * 80)
        print("test scenario:", example["scene"])
        print("user question:", question)
        response = agentic_agent.invoke({
            "messages": [{"role": "user", "content": question}],
        })
        called_tools = extract_called_tools(response)
        print("expected calls:", example["expected_tools"] or "no tool")
        print("actual calls:", called_tools or "no tool")
        print("route check:", "pass" if called_tools == example["expected_tools"] else "needs tuning")
        print("final answer:")
        print_agent_answer(response)

## 5. Agentic RAG summary

Agentic RAG does not replace ordinary RAG, it adds a "tool scheduling layer" once ordinary RAG is stable. Recommended order for teaching and rollout:

1. **Do ordinary RAG first**: make retrieval, prompts, and answer quality controllable;
2. **Then hybrid retrieval + rerank**: improve recall and ranking quality;
3. **Introduce the Agent last**: wrap the different retrieval chains, calculation tools, and business APIs as tools and let the model choose per task;
4. **Keep evaluating tool selection**: the core risk of an Agent is not whether the code runs, but whether it picks the right tool in the right situation.

The three core decisions summarise to: semantic questions go to simple retrieval, exact-keyword questions go to hybrid retrieval, questions independent of enterprise knowledge skip retrieval entirely.

In one sentence: ordinary RAG answers "how to answer from the knowledge base", Agentic RAG answers "whether to look it up, how to look it up, and whether another capability is better".


# Advanced RAG: evaluating quality

This chapter evaluates the hybrid retrieval RAG built earlier: first check whether the retrieval layer hits the right documents, then use LLM-as-Judge to compare the faithfulness, relevance, and completeness of the RAG answer against a no-knowledge-base baseline, and finally discuss the limitations and possible extensions.


## 1. Evaluating the retrieval layer and the generation layer separately

### 1.1 Retrieval metrics

**Related foundations:** the retrieval evaluation in the foundations module C06 (RAG system evaluation), lesson 01 (evaluation introduction). Compute Hit@K, MRR, page and section hits first, then run the existing faithfulness, relevance, and completeness scoring.


In [ ]:
def is_expected_document(doc, expected):
    metadata = doc.metadata
    source_name = metadata.get("source_name") or Path(str(metadata.get("source", ""))).name
    page_number = metadata.get("page_number", metadata.get("page"))
    section_title = metadata.get("section_title")
    checks = []
    if expected.get("source_contains"):
        checks.append(expected["source_contains"] in source_name)
    if expected.get("page_number") is not None:
        checks.append(page_number == expected["page_number"])
    if expected.get("section_title"):
        checks.append(section_title == expected["section_title"])
    return all(checks) if checks else False

def evaluate_retrieval_result(docs, expected, k=5):
    top_docs = docs[:k]
    ranks = [
        rank for rank, doc in enumerate(top_docs, 1)
        if is_expected_document(doc, expected)
    ]
    first_rank = min(ranks) if ranks else None
    return {
        f"hit@{k}": int(first_rank is not None),
        "reciprocal_rank": 1 / first_rank if first_rank else 0.0,
        "first_relevant_rank": first_rank,
    }

retrieval_eval_set = [
    {"question": "\u6570\u636e\u53ef\u9760\u6027\u5305\u62ec\u54ea\u4e9b\u5c42\uff1f", "expected": {"source_contains": "\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1", "page_number": 21, "section_title": "7.3 \u6570\u636e\u53ef\u9760\u6027"}},
    {"question": "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f", "expected": {"source_contains": "\u4e0a\u7f51\u65e5\u5fd7\u4e1a\u52a1"}},
    {"question": "\u4ec0\u4e48\u662f5GC\u4e1a\u52a1\uff1f", "expected": {"source_contains": "5GC\u4e1a\u52a1"}},
]
print(f"retrieval evaluation set ready: {len(retrieval_eval_set)} cases; extend it to 50 or more.")

### 1.2 Generation metrics

The generation layer uses three metrics - faithfulness, relevance, and completeness: faithfulness checks whether the answer is grounded in the context, relevance checks whether it answers the question directly, completeness checks whether it covers the key points of the reference answer. The LLM-as-Judge step later scores the RAG answer and the no-knowledge-base baseline separately.


## 2. End-to-end generation evaluation (LLM-as-Judge)


### 2.1 The evaluation plan

Once the RAG pipeline is assembled, the last step is to evaluate whether it is genuinely more reliable than a "bare LLM".

This chapter follows the **LLM-as-Judge** idea with a lightweight comparison experiment:

1. prepare 3 test questions in the VMAX domain, each with the key points of a reference answer;
2. reuse the query understanding, weighted hybrid recall, metadata filter, optional rerank, and context optimisation flow from the advanced hybrid retrieval chapter to generate the RAG answer;
3. call the same LLM directly with the same question to generate a no-knowledge-base baseline answer;
4. have another LLM score both on **faithfulness, relevance, and completeness** (1-5, 5 being best);
5. compare the two sets of scores in a table and observe how hybrid-retrieval RAG suppresses hallucination and improves factual accuracy.

> Note: no extra evaluation library (such as Ragas) is introduced here; the already-configured LLM acts as the judge so the notebook runs directly in the `env_rag` environment. The scores are indicative only - for real business use, combine them with human annotation.


### 2.2 Preparing the evaluation environment and the hybrid retriever


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

# evaluation environment: reuse the shared model configuration and connect to the hybrid vector store
import os
import httpx
import re
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from pymilvus import MilvusClient
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# model
eval_llm = my_llm
eval_emb = OpenAIEmbeddings(
    base_url=os.environ["XINFERENCE_OPENAI_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
    model=os.environ["XINFERENCE_EMBEDDING_MODEL_ID"],
)

connection_args = {"uri": os.environ["MILVUS_URI"]}
_eval_milvus_client = MilvusClient(**connection_args)

eval_hybrid_db = Milvus(
    embedding_function=eval_emb,
    builtin_function=BM25BuiltInFunction(output_field_names="sparse"),
    collection_name="ZXVMAXS_HYBRID",
    connection_args=connection_args,
    vector_field=["dense", "sparse"],
    text_field="text",
    consistency_level="Bounded",
)
# do not return raw vector fields during retrieval
eval_hybrid_db.fields = [
    field for field in eval_hybrid_db.fields
    if field not in {"dense", "sparse", "vector"}
]

def _format_eval_context(docs, preview_chars=500):
    lines = []
    for i, doc in enumerate(docs, 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "\u672a\u77e5\u6765\u6e90")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        section = doc.metadata.get("section_title", "\u672a\u8bc6\u522b\u7ae0\u8282")
        content = doc.page_content[:preview_chars].replace("\n", " ")
        lines.append(f"[{i}] source={source} | page={page} | section={section}\n{content}")
    return "\n\n".join(lines)

def retrieve_docs_for_eval(question: str, k: int = 20):
    """Reuse the query understanding, recall, filtering, rerank and deduplication flow from the hybrid retrieval chapter."""
    strategy = prepare_retrieval_query(question)
    normalized_question = strategy["normalized_query"]
    docs = eval_hybrid_db.similarity_search(
        normalized_question,
        k=k,
        ranker_type="weighted",
        ranker_params={"weights": [0.6, 0.4]},
    )
    business_type = strategy["business_type"]
    if business_type:
        # recall again with the business type attached, so filtering after the fact does not leave too few candidates.
        business_docs = eval_hybrid_db.similarity_search(
            f"{normalized_question} {business_type}",
            k=k,
            ranker_type="weighted",
            ranker_params={"weights": [0.6, 0.4]},
        )
        docs = deduplicate_documents([*docs, *business_docs])
        filtered_docs = [
            doc for doc in docs
            if doc.metadata.get("business_type") == business_type
        ]
        if filtered_docs:
            docs = filtered_docs

    if "xinference_qwen3_rerank" in globals():
        docs = xinference_qwen3_rerank(normalized_question, docs, top_n=8)
    return strategy, deduplicate_documents(docs)

def build_eval_context(question: str):
    strategy, docs = retrieve_docs_for_eval(question)
    adjacent_source_docs = globals().get("split_docs", [])
    if adjacent_source_docs:
        docs = expand_adjacent_chunks(docs[:3], adjacent_source_docs, window=1)
    docs = select_context_with_budget(docs, max_chars=6000)
    return strategy, docs, _format_eval_context(docs)

print("evaluation environment ready: query understanding -> hybrid retrieval -> metadata filter -> optional rerank -> context optimisation.")


### 2.3 Running the retrieval-layer evaluation


In [ ]:
retrieval_eval_rows = []
print(f"starting the retrieval-layer evaluation with {len(retrieval_eval_set)} questions.")

for index, sample in enumerate(retrieval_eval_set, 1):
    question = sample["question"]
    print(f"[{index}/{len(retrieval_eval_set)}] {question}", flush=True)
    strategy, docs = retrieve_docs_for_eval(question, k=20)
    metrics = evaluate_retrieval_result(docs, sample["expected"], k=5)
    retrieval_eval_rows.append({
        "question": question,
        "business_type": strategy["business_type"] or "unrestricted",
        "Hit@5": metrics["hit@5"],
        "MRR": round(metrics["reciprocal_rank"], 3),
        "first_relevant_rank": metrics["first_relevant_rank"] or "no hit",
    })
    print(
        f"    Hit@5={metrics['hit@5']} | MRR={metrics['reciprocal_rank']:.3f} "
        f"| first_rank={metrics['first_relevant_rank'] or 'no hit'}",
        flush=True,
    )

retrieval_eval_df = pd.DataFrame(retrieval_eval_rows)
display(retrieval_eval_df)
print("\nretrieval-layer average metrics:")
print(retrieval_eval_df[["Hit@5", "MRR"]].mean().round(3))

### 2.4 Building the RAG chain, the baseline chain, and the evaluation set


In [ ]:
# build the RAG QA chain and the baseline QA chain
rag_prompt = PromptTemplate.from_template("""
You are the VMAX operations assistant. Answer strictly from the context below.

Context:
{context}

Question: {question}

Answer requirements:
1. Answer only from the context; never invent information that is not in it;
2. If the context does not contain the answer, reply "I don't know";
3. Keep the answer concise and accurate, within 500 characters;
4. Do not output your reasoning process.

/no_think

Answer:""")

# retrieval runs explicitly inside the generation loop, so the steps can be printed and the same context reused for scoring.
rag_answer_chain = (
    rag_prompt
    | eval_llm
    | StrOutputParser()
)

baseline_prompt = PromptTemplate.from_template("""
Answer the question below directly and concisely, within 300 characters.
If you are unsure, say so explicitly; never invent specific vendors, products or features.
Do not output your reasoning process.

Question: {question}

/no_think
""")
baseline_answer_chain = baseline_prompt | eval_llm | StrOutputParser()

# evaluation dataset: question plus reference answer points
eval_set = [
    {
        "question": "ZXVMAX-S\u4e0a\u7f51\u65e5\u5fd7\u6709\u54ea\u4e9b\u529f\u80fd\uff1f",
        "reference": "\u4e0a\u7f51\u65e5\u5fd7\u4fdd\u5b58\u3001\u4e0a\u7f51\u65e5\u5fd7\u67e5\u8be2\u3001\u6279\u91cf\u5bfc\u5165\u67e5\u8be2\u3001\u65e5\u5fd7\u7ba1\u7406\u3001\u8d26\u53f7/\u89d2\u8272\u7ba1\u7406\u3001\u8d44\u6e90\u76d1\u63a7\u3001\u544a\u8b66\u7ba1\u7406\u3001\u7701\u7ea7\u7f51\u5173\u5bf9\u63a5\u3001\u62e8\u6d4b\u7ed3\u679c\u81ea\u52a8\u6bd4\u5bf9\u3001NAT\u65e5\u5fd7\u5165\u5e93\u3001\u5317\u5411\u63a5\u53e3\u3001\u4e0a\u7f51\u65e5\u5fd7\u5386\u53f2\u67e5\u8be2\u3001\u4e91\u5316\u4e0a\u7f51\u65e5\u5fd7XDR\u67e5\u8be2",
    },
    {
        "question": "\u4ec0\u4e48\u662fZXVMAX-S\u76845GC\u4e1a\u52a1\uff1f",
        "reference": "\u9762\u54115G\u6838\u5fc3\u7f51\u7684\u591a\u7ef4\u4ef7\u503c\u5206\u6790\u7cfb\u7edf\uff1b\u5305\u542b5G\u7cbe\u51c6\u5f00\u901a\u30015G\u4e1a\u52a1\u8fd0\u8425\u3001\u4e00\u952e\u6295\u8bc9\u3001\u533a\u57df\u611f\u77e5\u3001\u7528\u6237\u611f\u77e5\u3001\u5b9e\u65f6\u8ddf\u8e2a/\u4f1a\u8bdd\u8ddf\u8e2a\u7b49\u529f\u80fd",
    },
    {
        "question": "ZXVMAX-S\u5982\u4f55\u4fdd\u969c\u6570\u636e\u53ef\u9760\u6027\uff1f",
        "reference": "\u91c7\u96c6\u5c42\u91c7\u7528\u5197\u4f59\u78c1\u76d8\u9635\u5217\u5e76\u652f\u6301\u8865\u5145\u91c7\u96c6\uff1b\u5b58\u50a8\u5171\u4eab\u5c42\u4f7f\u7528MPP\u6570\u636e\u5e93\u591a\u526f\u672c\u3001\u5197\u4f59\u78c1\u76d8\u9635\u5217\u548c\u53cc\u63a7\u5236\u5668\uff1b\u5e94\u7528\u5206\u6790\u5c42\u7684\u8f6f\u4ef6\u4e0e\u914d\u7f6e\u6570\u636e\u90e8\u7f72\u5728\u5197\u4f59\u78c1\u76d8\u9635\u5217\u4e0a\uff0c\u786e\u4fdd\u6545\u969c\u65f6\u6570\u636e\u4e0d\u4e22\u5931",
    },
]

print(f"preparing {len(eval_set)} evaluation questions")


### 2.5 Generating the RAG and baseline answers


In [ ]:
# generate the answers (print retrieval, RAG and baseline progress question by question)
import time

total_questions = len(eval_set)
generation_started_at = time.perf_counter()

for index, item in enumerate(eval_set, 1):
    question = item["question"]
    question_started_at = time.perf_counter()
    print("=" * 80, flush=True)
    print(f"[{index}/{total_questions}] question: {question}", flush=True)

    print("  1. running hybrid retrieval and context optimisation...", flush=True)
    retrieval_started_at = time.perf_counter()
    strategy, docs, context = build_eval_context(question)
    item["retrieval_strategy"] = strategy
    item["retrieved_docs"] = docs
    item["retrieved_context"] = context
    print(
        f"     done: route=hybrid | business_type={strategy['business_type'] or 'unrestricted'} "
        f"| final context={len(docs)} chunks/{len(context)} characters "
        f"| elapsed={time.perf_counter() - retrieval_started_at:.1f}s",
        flush=True,
    )
    for doc_index, doc in enumerate(docs[:5], 1):
        source = (doc.metadata.get("source_name") or doc.metadata.get("source")
                  or doc.metadata.get("file_path") or "\u672a\u77e5\u6765\u6e90")
        section = doc.metadata.get("section_title", "\u672a\u8bc6\u522b\u7ae0\u8282")
        page = doc.metadata.get("page_number", doc.metadata.get("page", "-"))
        print(f"       [{doc_index}] {source} | page={page} | section={section}", flush=True)

    print("  2. generating the RAG answer...", flush=True)
    rag_started_at = time.perf_counter()
    item["rag_answer"] = rag_answer_chain.invoke({
        "context": context,
        "question": question,
    })
    print(f"     done, elapsed={time.perf_counter() - rag_started_at:.1f}s", flush=True)
    print("     RAG preview: ", item["rag_answer"][:160].replace("\n", " "), flush=True)

    print("  3. generating the no-knowledge-base baseline answer...", flush=True)
    baseline_started_at = time.perf_counter()
    item["baseline_answer"] = baseline_answer_chain.invoke({"question": question})
    print(f"     done, elapsed={time.perf_counter() - baseline_started_at:.1f}s", flush=True)
    print("     baseline preview: ", item["baseline_answer"][:160].replace("\n", " "), flush=True)
    print(f"  total time for this question: {time.perf_counter() - question_started_at:.1f}s", flush=True)

print("=" * 80)
print(f"RAG and baseline answers generated, total time: {time.perf_counter() - generation_started_at:.1f}s")


### 2.6 Scoring with LLM-as-Judge


In [ ]:
# LLM-as-Judge: score the answers on three dimensions
judge_prompt = PromptTemplate.from_template("""
You are an impartial evaluation expert. Using the context, the question and the reference answer points below, score the assistant's answer.

[context]
{context}

[question]
{question}

[reference answer points]
{reference}

[assistant answer]
{answer}

Score the answer on the three dimensions below (1-5, 5 best) and give a short reason:
1. Faithfulness: is every claim in the answer supported by the context.
2. Relevance: does the answer address the question instead of going off topic.
3. Completeness: does the answer cover the reference answer points.

Output strictly in this format:
faithfulness: <score>
relevance: <score>
completeness: <score>
reason: <one-sentence reason>

Do not output your reasoning process.
/no_think
""")

judge_chain = judge_prompt | eval_llm | StrOutputParser()

def parse_judge_result(text: str):
    result = {"faithfulness": 0, "relevance": 0, "completeness": 0, "reason": ""}
    faith = re.search("faithfulness[:\uff1a]\s*(\d+)", text)
    rel = re.search("relevance[:\uff1a]\s*(\d+)", text)
    comp = re.search("completeness[:\uff1a]\s*(\d+)", text)
    reason = re.search("reason[:\uff1a]\s*(.+)", text, re.DOTALL)
    if faith:
        result["faithfulness"] = int(faith.group(1))
    if rel:
        result["relevance"] = int(rel.group(1))
    if comp:
        result["completeness"] = int(comp.group(1))
    if reason:
        result["reason"] = reason.group(1).strip().replace("\n", " ")
    return result

total_judgements = len(eval_set) * 2
judgement_index = 0
judge_started_at = time.perf_counter()

for question_index, item in enumerate(eval_set, 1):
    context = item["retrieved_context"]
    for key, label in [("rag_answer", "RAG"), ("baseline_answer", "baseline")]:
        judgement_index += 1
        score_started_at = time.perf_counter()
        print(
            f"[{judgement_index}/{total_judgements}] scoring: question {question_index} / {label}",
            flush=True,
        )
        judge_text = judge_chain.invoke({
            "context": context,
            "question": item["question"],
            "reference": item["reference"],
            "answer": item[key],
        })
        item[f"{key}_score"] = parse_judge_result(judge_text)
        score = item[f"{key}_score"]
        print(
            f"    faithfulness={score['faithfulness']} | relevance={score['relevance']} "
            f"| completeness={score['completeness']} | elapsed={time.perf_counter() - score_started_at:.1f}s",
            flush=True,
        )

print(f"scoring finished, total time: {time.perf_counter() - judge_started_at:.1f}s")


### 2.7 Aggregating and comparing the evaluation results


In [ ]:
# summarise the comparison
rows = []
for item in eval_set:
    rows.append({
        "question": item["question"],
        "approach": "RAG",
        "faithfulness": item["rag_answer_score"]["faithfulness"],
        "relevance": item["rag_answer_score"]["relevance"],
        "completeness": item["rag_answer_score"]["completeness"],
        "average": round(sum([item["rag_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })
    rows.append({
        "question": item["question"],
        "approach": "baseline (no RAG)",
        "faithfulness": item["baseline_answer_score"]["faithfulness"],
        "relevance": item["baseline_answer_score"]["relevance"],
        "completeness": item["baseline_answer_score"]["completeness"],
        "average": round(sum([item["baseline_answer_score"][k] for k in ["faithfulness", "relevance", "completeness"]]) / 3, 2),
    })

df = pd.DataFrame(rows)
display(df)

print("\n===== average score comparison =====")
print(df.groupby("approach")[["faithfulness", "relevance", "completeness", "average"]].mean().round(2))

print("\n===== example of the detailed reasons (first question) =====")
print("question:", eval_set[0]["question"])
print("\n[RAG answer]")
print(eval_set[0]["rag_answer"][:300])
print("reason:", eval_set[0]["rag_answer_score"]["reason"])
print("\n[baseline answer]")
print(eval_set[0]["baseline_answer"][:300])
print("reason:", eval_set[0]["baseline_answer_score"]["reason"])


## 3. Evaluation conclusions and limitations

### 3.1 Reading the evaluation results

Through this simple LLM-as-Judge experiment you can usually observe that:

- **RAG**: faithfulness and completeness are usually clearly higher than the baseline, because the answer comes from the local knowledge base;
- **Baseline**: prone to answers that "look professional but are fabricated" (for example claiming ZXVMAX-S is a Huawei product), so the hallucination risk is higher;
- **Relevance**: when the question is clearly phrased both are usually fine; but when the question needs the exact keywords in the documents, RAG is more stable.

### 3.2 Limitations and further optimisation

1. **The LLM judge is biased too**: the scoring model may prefer "longer" or "more complete" answers, so treat the results as indicative;
2. **Small sample**: only 3 questions are used here; for real business prepare an evaluation set of 50+ items;
3. **Retrieval metrics are still basic**: we only demo Hit@K, MRR, page and section hits; later add Recall@K, NDCG, and per-business statistics;
4. **You can bring in Ragas**: for something more systematic, install `ragas` and use built-in metrics such as `faithfulness`, `answer_relevancy`, and `context_recall`.

This chapter moves RAG from "it runs" to "it can be evaluated", making the whole notebook a more complete loop.


## 4. Further reading: the Ragas evaluation framework

**Ragas** (Retrieval-Augmented Generation Assessment) is an open-source evaluation framework designed for RAG systems, quantifying both the retrieval and the generation stages without requiring manually annotated reference answers.

### 4.1 Core metrics

- **Faithfulness**: measures whether the generated answer is grounded in the provided context, detecting whether the model fabricated information that is not in the context.

- **Answer Relevancy**: measures how relevant the generated answer is to the question, penalising answers that miss the question or contain irrelevant filler.

- **Context Relevancy**: measures how relevant the retrieved document chunks are to the question, that is the proportion of irrelevant content removed.

- **Context Recall**: measures whether the retrieved context contains all the information needed to answer the question.

### 4.2 Why use Ragas

1. **Automated**: no need to prepare reference answers by hand, which suits fast iteration of the RAG pipeline;
2. **End to end**: covers retrieval and generation together, making it easy to tell whether recall is insufficient or generation is inaccurate;
3. **Friendly LangChain integration**: you can pass LangChain's `retriever` and `chain` results straight in for scoring.

### 4.3 How to adopt it

In a real business setting, first get the evaluation flow working with the LLM-as-Judge method in this section, then migrate to Ragas step by step. The typical steps are: prepare 50+ evaluation questions -> use LangChain to collect questions and answers -> compute the average scores with the built-in Ragas metrics -> locate the optimisation direction by splitting "retrieval metrics" from "generation metrics".

With Ragas in place, a RAG system is no longer just "able to answer" - it can keep getting measurably better at answering.
